In [ ]:
!pip uninstall -y bitsandbytes
!pip install -q bitsandbytes>=0.41.0 transformers>=4.35.0 accelerate

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/')

In [ ]:
import gc
import logging
import time
from getpass import getpass
from pathlib import Path
from typing import Optional
import sys
import json

import pandas as pd
import requests
import torch
from requests.adapters import HTTPAdapter
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from urllib3.util.retry import Retry
import re

OUTPUT_DIR = "/content/gdrive/My Drive/Agile/"
DATA_FILE = "peft_102_questions_answers.csv"
API_URL = "https://api.vsellm.ru/v1/chat/completions"
API_KEY = getpass("API Key: ")

API_MODELS = [
    "gemini-3-pro-preview",
    "anthropic/claude-sonnet-4.6",
]

HF_MODELS = [
    "AvitoTech/avibe",
    "t-tech/T-lite-it-1.0",
]

MAX_RETRIES = 3
TIMEOUT = 120
RETRY_DELAY = 5
TEMPERATURE = 0.7
MAX_TOKENS = 4096

In [ ]:
root_logger = logging.getLogger()
root_logger.handlers.clear()
root_logger.setLevel(logging.INFO)

handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
root_logger.addHandler(handler)

logger = logging.getLogger(__name__)

# Генерация ответов

In [ ]:
AGILE_COACH_PROMPT = """### РОЛЬ ###

Ты опытный Agile коуч, который помогает командам в корпоративном
мессенджере Mattermost. Ты ответственен за качество и применимость
своих советов.

---

### ОСНОВНЫЕ ИНСТРУКЦИИ ###

* **Практичность прежде всего:** Твоя главная цель — дать полный
  и применимый совет. Каждый тезис должен быть логически обоснован
  и чётко объяснён. Совет, который звучит убедительно, но основан
  на неверных предпосылках или неприменим в реальной команде,
  считается провалом.

* **Честность в отношении полноты:** Если вопрос слишком общий
  или тебе не хватает контекста для полноценного совета, ты
  **не должен** конструировать ответ, который выглядит экспертным,
  но содержит скрытые изъяны или неоправданные допущения. Вместо
  этого дай только тот совет, который ты можешь обосновать. Частичный
  совет считается ценным, если он представляет собой реальный шаг
  вперёд. Примеры значимых частичных ответов:
  * Выявление корневой причины проблемы без готового решения.
  * Разбор одного конкретного аспекта при невозможности охватить всё.
  * Формулировка уточняющего вопроса, который сам по себе помогает
    команде переосмыслить ситуацию.
  * Для задач на оптимизацию процессов — указание на ограничение
    без навязывания конкретного инструмента.
  * Если ключевого контекста не хватает (методология команды,
    размер команды, роль сотрудника и т.п.) — задай ОДИН
    уточняющий вопрос сотруднику вместо совета. Выбирай тот вопрос,
    ответ на который максимально сузит неопределённость.

* **Тон и формат:** Отвечай коротко и по делу — 2-4 предложения
  или список из 2-3 пунктов. Если сообщение содержит приветствие
  (привет, добрый день, здравствуй и т.п.) — ОБЯЗАТЕЛЬНО начни ответ
  с ответного приветствия одной фразой, затем дай совет.
  Стиль: дружелюбный, профессиональный, без эмодзи, без академизма.
  Используй ТОЛЬКО русский язык.

---

### ФОРМАТ ВЫВОДА ###

Твой ответ ДОЛЖЕН быть структурирован в следующие разделы строго
в указанном порядке.

**1. Резюме**
Этот раздел формируется внутри, до написания финального ответа,
и содержит две части:

* **а. Вердикт:** Чётко определи, можешь ли ты дать полный
  или частичный совет.
  * **Для полного совета:** Сформулируй главный тезис, например:
    "Я могу дать конкретный совет. Главное действие: ..."
  * **Для частичного совета:** Укажи, что именно ты можешь обосновать,
    например: "Контекста недостаточно для полного ответа, но я могу
    точно сказать, что ..."

* **б. Набросок ответа:** Составь высокоуровневый план совета.
  Он должен включать:
  * Описание общей логики рекомендации.
  * Ключевые тезисы с опорой на конкретные Agile-практики.
  * Если применимо — разбор случаев или альтернативных сценариев.

**2. Финальный ответ**
Напиши итоговый совет. Он должен содержать ТОЛЬКО полезные,
обоснованные рекомендации — без внутренних рассуждений вслух,
альтернативных вариантов, которые ты отверг, и неудачных формулировок.

---

### ИНСТРУКЦИЯ ПО САМОПРОВЕРКЕ ###

Прежде чем финализировать ответ, тщательно перечитай "Набросок ответа"
и "Финальный ответ". Для каждого тезиса пройди по чеклисту:

**а. Проверка на критическую ошибку:**
Критическая ошибка — это любой тезис, нарушающий логику совета.
Сюда входят:
* **Логические ошибки** — например, вывод "команда демотивирована,
  значит нужно больше митингов" без причинно-следственного обоснования.
* **Фактические ошибки** — например, приписывание практики не той
  методологии, к которой она относится.

Если критическая ошибка найдена:
* Зафиксируй, что данный тезис **делает совет недействительным**.
* Не развивай аргументацию, опирающуюся на этот тезис.
* Проверь, есть ли в ответе независимые части, которые остаются
  верными.

**б. Проверка на пробел в обосновании:**
Пробел в обосновании — это тезис, который может быть верным,
но сформулирован расплывчато, бездоказательно или без опоры
на конкретную практику.

Если пробел найден:
* Либо подкрепи тезис конкретным обоснованием.
* Либо явно обозначь его как предположение, требующее уточнения
  контекста.
* Затем продолжи проверку остальных тезисов.

**в. Проверка формата:**
* Если в вопросе есть приветствие — первая строка финального ответа
  содержит ответное приветствие?
* Если нет — добавь перед советом.

---

### ПРИМЕР ВНУТРЕННЕЙ САМОПРОВЕРКИ ###

*(Это иллюстрация процесса. В финальный ответ не включается.)*

Черновой тезис: "Проведи ретроспективу и всё наладится."

Проверка:
* **Место:** "и всё наладится"
  * **Проблема:** Критическая ошибка — причинно-следственная связь
    не обоснована. Ретроспектива — инструмент рефлексии, а не
    универсальное решение любой командной проблемы.

Исправленный тезис: "Проведи ретроспективу, сфокусировав её на одной
конкретной проблеме — это даст команде структуру для обсуждения
и повысит шанс выйти с actionable-решением."

### ПРИМЕР ОТВЕТА НА ПРИВЕТСТВИЕ ###

Вопрос сотрудника: "Привет! Как провести ретроспективу?"

Финальный ответ:
Привет!
Начни ретроспективу с выбора формата — например, Start/Stop/Continue
или 4Ls. Зафиксируй не более 2-3 actionable-пунктов в конце встречи
и назначь ответственных за каждый. Без конкретных договорённостей
ретроспектива превращается в разговор ни о чём.

### ПРИМЕР УТОЧНЯЮЩЕГО ВОПРОСА ###

Вопрос сотрудника: "Как улучшить работу команды?"

Финальный ответ:
Чтобы дать точный совет, уточни: в какой методологии работает
команда — Scrum, Kanban или что-то другое?

---

### ВОПРОС СОТРУДНИКА ###

{question}

---

### НАПОМИНАНИЕ О ЗАДАЧЕ ###

Твоя задача — действовать как Agile коуч в быстрой переписке.
Сформируй внутренний набросок, пройди по чеклисту самопроверки,
устрани критические ошибки и пробелы в обосновании, затем выдай
только финальный ответ на русском языке."""

In [ ]:
class ApiGenerator:
    def __init__(
        self,
        api_url: str,
        api_key: str,
        max_retries: int = 3,
        timeout: int = 120,
        retry_delay: int = 5,
    ) -> None:
        self.api_url = api_url
        self.api_key = api_key
        self.max_retries = max_retries
        self.timeout = timeout
        self.retry_delay = retry_delay

    def _create_session(self) -> requests.Session:
        session = requests.Session()
        retry = Retry(
            total=self.max_retries,
            backoff_factor=1,
            status_forcelist=[429, 500, 502, 503, 504],
            allowed_methods=["POST"],
        )
        adapter = HTTPAdapter(max_retries=retry)
        session.mount("http://", adapter)
        session.mount("https://", adapter)
        return session

    def generate(
        self,
        model: str,
        messages: list,
        temperature: float = 0.7,
        max_tokens: int = 1024,
    ) -> Optional[str]:
        headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {self.api_key}",
        }
        payload = {
            "model": model,
            "messages": messages,
            "temperature": temperature,
            "max_tokens": max_tokens,
        }
        session = self._create_session()

        for attempt in range(self.max_retries):
            try:
                response = session.post(
                    self.api_url,
                    headers=headers,
                    json=payload,
                    timeout=self.timeout,
                )
                response.raise_for_status()
                result = response.json()
                message = result.get("choices", [{}])[0].get("message", {})
                content = message.get("content", "").strip()
                if not content:
                    content = message.get("reasoning_content", "").strip()
                    if content:
                        logger.warning(f"{model}: using reasoning_content as fallback")
                if content:
                    return content
            except Exception as e:
                logger.warning(f"Attempt {attempt + 1} failed for {model}: {e}")
                time.sleep(self.retry_delay * (attempt + 1))

        logger.error(f"All attempts failed for {model}")
        return None

    def generate_for_dataframe(
        self,
        df: pd.DataFrame,
        model: str,
        prompt_template: str,
        temperature: float,
        max_tokens: int,
    ) -> tuple:
        answers = []
        errors = 0
        for _, row in tqdm(df.iterrows(), total=len(df), desc=model):
            prompt = prompt_template.format(question=row["question"])
            messages = [{"role": "user", "content": prompt}]
            answer = self.generate(
                model=model,
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens,
            )
            if answer:
                answers.append(answer)
            else:
                answers.append("[ERROR: No response]")
                errors += 1
            time.sleep(0.5)
        return answers, errors

In [ ]:
class HFGenerator:
    def __init__(
        self,
        model_name: str,
        temperature: float = 0.7,
        max_tokens: int = 1024,
        use_quantization: bool = False,
    ) -> None:
        self.model_name = model_name
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.use_quantization = use_quantization
        self._model = None
        self._tokenizer = None

    def _load(self) -> None:
        logger.info(f"Loading {self.model_name}")
        self._tokenizer = AutoTokenizer.from_pretrained(
            self.model_name, trust_remote_code=True
        )
        if self.use_quantization:
            try:
                quantization_config = BitsAndBytesConfig(
                    load_in_8bit=True,
                    llm_int8_threshold=6.0,
                    llm_int8_has_fp16_weight=False,
                )
                self._model = AutoModelForCausalLM.from_pretrained(
                    self.model_name,
                    quantization_config=quantization_config,
                    device_map="auto",
                    trust_remote_code=True,
                )
                logger.info(f"{self.model_name} loaded with 8-bit quantization")
                return
            except Exception as e:
                logger.warning(f"8-bit quantization failed: {e}. Falling back to FP16")

        self._model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            device_map="auto",
            torch_dtype=torch.float16,
            trust_remote_code=True,
        )
        logger.info(f"{self.model_name} loaded with FP16")

    def _unload(self) -> None:
        del self._model
        del self._tokenizer
        self._model = None
        self._tokenizer = None
        torch.cuda.empty_cache()
        gc.collect()
        logger.info(f"{self.model_name} unloaded from memory")

    def _generate_single(self, prompt: str) -> str:
        inputs = self._tokenizer(prompt, return_tensors="pt").to(self._model.device)
        with torch.no_grad():
            outputs = self._model.generate(
                **inputs,
                max_new_tokens=self.max_tokens,
                temperature=self.temperature,
                do_sample=self.temperature > 0,
                top_p=0.9,
                pad_token_id=self._tokenizer.eos_token_id,
            )
        generated = self._tokenizer.decode(outputs[0], skip_special_tokens=True)
        del inputs, outputs
        torch.cuda.empty_cache()
        return generated

    @staticmethod
    def _extract_response(generated_text: str, prompt: str) -> str:
        if "Assistant:" in generated_text:
            return generated_text.split("Assistant:")[-1].strip()
        if generated_text.startswith(prompt):
            return generated_text[len(prompt):].strip()
        return generated_text.strip()

    def generate_for_dataframe(
        self,
        df: pd.DataFrame,
        prompt_template: str,
    ) -> tuple:
        self._load()
        answers = []
        errors = 0
        for idx, row in tqdm(df.iterrows(), total=len(df), desc=self.model_name):
            try:
                prompt = prompt_template.format(question=row["question"])
                full_prompt = f"User: {prompt}\n\nAssistant:"
                generated = self._generate_single(full_prompt)
                response = self._extract_response(generated, full_prompt)
                answers.append(response)
            except Exception as e:
                logger.error(f"Error on row {idx}: {e}")
                answers.append("[ERROR: No response]")
                errors += 1
        self._unload()
        return answers, errors

In [ ]:
class GenerationPipeline:
    def __init__(
        self,
        data_path: Path,
        output_dir: Path,
        prompt_template: str,
        api_generator: ApiGenerator,
        temperature: float = 0.7,
        max_tokens: int = 1024,
    ) -> None:
        self.data_path = data_path
        self.output_dir = output_dir
        self.prompt_template = prompt_template
        self.api_generator = api_generator
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.df: Optional[pd.DataFrame] = None

    def load_data(self) -> None:
        self.df = pd.read_csv(self.data_path, sep=";")
        self.df = self.df.replace(r"\n", " ", regex=True)
        #self.df = self.df.head(3) # for testing
        logger.info(f"Loaded {len(self.df)} rows. Columns: {list(self.df.columns)}")

    @staticmethod
    def _column_name(model_name: str) -> str:
        return f"{model_name.replace('/', '_')}_answer"

    def _save_checkpoint(self, model_name: str) -> None:
        path = self.output_dir / f"checkpoint_{model_name.replace('/', '_')}.csv"
        self.df.to_csv(path, index=False, sep=";")
        logger.info(f"Checkpoint saved: {path}")

    @staticmethod
    def _log_stats(model_name: str, answers: list, errors: int) -> None:
        total = len(answers)
        success = total - errors
        rate = (success / total * 100) if total > 0 else 0.0
        logger.info(f"{model_name}: {success}/{total} ({rate:.1f}%)")

    def run_api_models(self, models: list) -> None:
        logger.info("Generating responses: API models")
        for model in models:
            col = self._column_name(model)
            if col in self.df.columns:
                logger.info(f"{col} already exists, skipping")
                continue
            answers, errors = self.api_generator.generate_for_dataframe(
                df=self.df,
                model=model,
                prompt_template=self.prompt_template,
                temperature=self.temperature,
                max_tokens=self.max_tokens,
            )
            self.df[col] = answers
            self._save_checkpoint(model)
            self._log_stats(model, answers, errors)

    def run_hf_models(self, models: list) -> None:
        logger.info("Generating responses: HF models")
        for model_name in models:
            col = self._column_name(model_name)
            if col in self.df.columns:
                logger.info(f"{col} already exists, skipping")
                continue
            try:
                generator = HFGenerator(
                    model_name=model_name,
                    temperature=self.temperature,
                    max_tokens=self.max_tokens,
                )
                answers, errors = generator.generate_for_dataframe(
                    df=self.df,
                    prompt_template=self.prompt_template,
                )
            except Exception as e:
                logger.error(f"Failed to run {model_name}: {e}")
                answers = ["[ERROR: Model failed]"] * len(self.df)
                errors = len(self.df)
            self.df[col] = answers
            self._save_checkpoint(model_name)
            self._log_stats(model_name, answers, errors)

    def save_final(self) -> None:
        path = self.output_dir / "results_final.csv"
        self.df.to_csv(path, index=False, sep=";")
        logger.info(f"Final results saved: {path}")

    def parse_responses(self) -> None:
        answer_cols = [c for c in self.df.columns if c.endswith("_answer")]
        marker_pattern = re.compile(
            r"\*{0,2}2\.\s*\*{0,2}Финальный ответ\*{0,2}[:\s]*",
            re.IGNORECASE
        )

        def split_response(text: str) -> tuple:
            if not isinstance(text, str):
                return "", text
            parts = marker_pattern.split(text, maxsplit=1)
            if len(parts) == 2:
                final = re.sub(r"^[\*\s]+", "", parts[1])
                return parts[0].strip(), final.strip()
            return "", text

        for col in answer_cols:
            draft_col = col.replace("_answer", "_draft")
            final_col = col.replace("_answer", "_final")
            self.df[draft_col], self.df[final_col] = zip(
                *self.df[col].apply(split_response)
            )

        logger.info(f"Parsed {len(answer_cols)} columns into _draft and _final")

In [ ]:
output_dir = Path(OUTPUT_DIR)
data_path = output_dir / DATA_FILE

api_generator = ApiGenerator(
    api_url=API_URL,
    api_key=API_KEY,
    max_retries=MAX_RETRIES,
    timeout=TIMEOUT,
    retry_delay=RETRY_DELAY,
)

pipeline = GenerationPipeline(
    data_path=data_path,
    output_dir=output_dir,
    prompt_template=AGILE_COACH_PROMPT,
    api_generator=api_generator,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
)

In [ ]:
pipeline.load_data()

In [ ]:
pipeline.run_api_models(API_MODELS)

In [ ]:
pipeline.run_hf_models(HF_MODELS)

In [ ]:
pipeline.parse_responses()

In [ ]:
display(pipeline.df.head(10))

In [ ]:
display(pipeline.df.head(10))

In [ ]:
pipeline.save_final()

# LLM-as-a-judge

In [ ]:
ANSWER_MODELS = [
    "gemini-3-pro-preview",
    "anthropic/claude-sonnet-4.6",
    "AvitoTech/avibe",
    "t-tech/T-lite-it-1.0",
]

CRITERIA = [
    "agile_correctness",
    "practicality",
    "context_handling",
    "politeness",
    "text_cleanliness",
    "language_quality",
]

In [ ]:
RESULTS_FILE = "results_final.csv"
JUDGE_MODEL = "openai/gpt-5.2"

EVAL_ROWS = 20
MAX_RETRIES = 3
TIMEOUT = 120
RETRY_DELAY = 5

In [ ]:
JUDGE_CRITERIA = """
Ты — строгий проверяющий. Твоя задача — оценить ответ Agile коуча
по 6 критериям. Каждый критерий бинарный: 0 или 1.

### ОСНОВНЫЕ ИНСТРУКЦИИ ###

* Ты действуешь исключительно как **верификатор**, а не как советчик.
  Не исправляй ответ и не предлагай улучшений.
* Проверяй ответ **шаг за шагом** по каждому критерию.
* Для каждого нарушения **цитируй конкретный фрагмент** ответа,
  который стал основанием для оценки 0.
* Не допускай промежуточных значений: только 0 или 1.
  Если есть сомнение — ставь 0.

---

### КРИТЕРИИ ###

**1. agile_correctness — Корректность с точки зрения Agile**

1 — совет полностью соответствует практикам и принципам Agile:
  - каждая упомянутая практика приписана правильной методологии
  - нет утверждений, противоречащих Agile Manifesto или Scrum Guide
  - нет логических ошибок вида "X сломано, значит делай Y"
    без причинно-следственного обоснования

0 — хотя бы одно из:
  - практика приписана неверной методологии
  - совет прямо противоречит базовым принципам Agile
  - причинно-следственная связь между проблемой и рекомендацией
    отсутствует или ложная

---

**2. practicality — Практическая применимость**

1 — ответ содержит конкретное действие, выполнимое в реальной
    корпоративной команде без дополнительных ресурсов:
  - есть понятный следующий шаг
  - рекомендация не требует идеальных условий

0 — хотя бы одно из:
  - совет абстрактный, без конкретных шагов
    (пример нарушения: "улучшите коммуникацию в команде")
  - рекомендация применима только в идеальных условиях
  - непонятно, что именно нужно сделать после прочтения

---

**3. context_handling — Работа с неполным контекстом**

1 — модель корректно обработала уровень полноты вопроса:
  - если контекст достаточен: дан конкретный совет
  - если контекст недостаточен: модель явно обозначила,
    чего не хватает, и либо задала уточняющий вопрос,
    либо дала совет для наиболее вероятного сценария
    с явным указанием допущения

0 — хотя бы одно из:
  - модель сделала неоправданные допущения без их обозначения
  - при явно неполном вопросе дан уверенный совет без оговорок
  - при достаточном контексте модель уклонилась от конкретики

---

**4. politeness — Вежливость и тон**

1 — тон соответствует дружелюбному профессиональному стилю:
  - если в вопросе есть приветствие — ответ начинается
    с ответного приветствия
  - нет покровительственного, холодного или фамильярного тона

0 — хотя бы одно из:
  - сотрудник поздоровался, модель проигнорировала приветствие
  - тон звучит пренебрежительно или менторски
    (пример нарушения: "Вы должны понимать, что...")
  - тон избыточно фамильярный

---

**5. text_cleanliness — Чистота текста**

1 — текст не содержит артефактов:
  - нет эмодзи
  - нет висящих символов (* ** :) в начале текста
  - нет технических артефактов генерации (повторов, обрывов,
    незакрытых скобок)

0 — хотя бы одно из:
  - присутствуют эмодзи
  - текст начинается с ** или :
  - есть артефакты генерации

---

**6. language_quality — Грамотность и языковая чистота**

1 — текст написан грамотно на русском языке:
  - нет орфографических и грубых пунктуационных ошибок
  - англицизмы используются только там, где нет устоявшегося
    русского эквивалента (спринт, бэклог, скрам — допустимо;
    "имплементировать" вместо "внедрить" — нарушение)

0 — хотя бы одно из:
  - есть орфографические или грубые грамматические ошибки
  - текст перегружен английскими словами без необходимости
  - используются слова там, где они неуместны
    (пример нарушения: "эджайл-подход" вместо "Agile-подход")

---

### ИНСТРУКЦИЯ ПО ВЕРИФИКАЦИИ ###

Перед выводом результата выполни следующие шаги:

Шаг 1. Прочитай вопрос сотрудника и финальный ответ модели полностью.

Шаг 2. Для каждого критерия:
  - процитируй фрагмент ответа, на который опираешься
  - зафикси: выполнен критерий или нарушен
  - если нарушен — укажи конкретную причину

Шаг 3. Проверь согласованность оценок: если agile_correctness = 0,
  то practicality, скорее всего, тоже 0 — убедись, что оценки
  не противоречат друг другу.

Шаг 4. Сформируй итоговый JSON.

---

### ФОРМАТ ВЫВОДА ###

Верни результат строго в формате JSON без пояснений вне блока:
```json
{{
  "agile_correctness": <0 или 1>,
  "practicality": <0 или 1>,
  "context_handling": <0 или 1>,
  "politeness": <0 или 1>,
  "text_cleanliness": <0 или 1>,
  "language_quality": <0 или 1>,
  "total": <сумма баллов от 0 до 6>,
  "violations": [
    {{
      "criterion": "<название критерия>",
      "quote": "<цитата из ответа>",
      "reason": "<причина нарушения>"
    }}
  ]
}}
```

Если нарушений нет — "violations" возвращается пустым списком [].

---

### ДАННЫЕ ДЛЯ ОЦЕНКИ ###

Вопрос сотрудника:
{question}

Финальный ответ модели:
{answer}
"""

In [ ]:
class JudgeClient:
    def __init__(
        self,
        api_url: str,
        api_key: str,
        model: str,
        max_retries: int = 3,
        timeout: int = 120,
        retry_delay: int = 5,
    ) -> None:
        self.api_url = api_url
        self.api_key = api_key
        self.model = model
        self.max_retries = max_retries
        self.timeout = timeout
        self.retry_delay = retry_delay

    def _create_session(self) -> requests.Session:
        session = requests.Session()
        retry = Retry(
            total=self.max_retries,
            backoff_factor=1,
            status_forcelist=[429, 500, 502, 503, 504],
            allowed_methods=["POST"],
        )
        adapter = HTTPAdapter(max_retries=retry)
        session.mount("http://", adapter)
        session.mount("https://", adapter)
        return session

    def evaluate(self, question: str, answer: str, prompt_template: str) -> Optional[dict]:
        prompt = prompt_template.format(question=question, answer=answer)
        headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {self.api_key}",
        }
        payload = {
            "model": self.model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.0,
        }
        session = self._create_session()

        for attempt in range(self.max_retries):
            try:
                response = session.post(
                    self.api_url,
                    headers=headers,
                    json=payload,
                    timeout=self.timeout,
                )
                response.raise_for_status()
                content = (
                    response.json()
                    .get("choices", [{}])[0]
                    .get("message", {})
                    .get("content", "")
                    .strip()
                )
                return self._parse_json(content)
            except Exception as e:
                logger.warning(f"Attempt {attempt + 1} failed: {e}")
                time.sleep(self.retry_delay * (attempt + 1))

        logger.error(f"All attempts failed for judge call")
        return None

    @staticmethod
    def _parse_json(content: str) -> Optional[dict]:
        clean = re.sub(r"```json|```", "", content).strip()
        try:
            return json.loads(clean)
        except json.JSONDecodeError as e:
            logger.error(f"JSON parse error: {e}. Content: {content[:200]}")
            return None

In [ ]:
class JudgePipeline:
    def __init__(
        self,
        results_path: Path,
        output_dir: Path,
        judge_client: JudgeClient,
        prompt_template: str,
        answer_models: list,
        criteria: list,
        eval_rows: int = 20,
    ) -> None:
        self.results_path = results_path
        self.output_dir = output_dir
        self.judge_client = judge_client
        self.prompt_template = prompt_template
        self.answer_models = answer_models
        self.criteria = criteria
        self.eval_rows = eval_rows
        self.df: Optional[pd.DataFrame] = None
        self.results: dict[str, pd.DataFrame] = {}

    def load_data(self) -> None:
        self.df = pd.read_csv(self.results_path, sep=";").head(self.eval_rows)
        logger.info(f"Loaded {len(self.df)} rows from {self.results_path}")

    @staticmethod
    def _model_to_col(model_name: str, suffix: str) -> str:
        return f"{model_name.replace('/', '_')}_{suffix}"

    def _empty_criteria_row(self) -> dict:
        return {c: None for c in self.criteria}

    def _evaluate_row(self, question: str, answer: str) -> dict:
        if not isinstance(answer, str) or answer.startswith("[ERROR"):
            logger.warning("Skipping invalid answer")
            return self._empty_criteria_row()

        result = self.judge_client.evaluate(
            question=question,
            answer=answer,
            prompt_template=self.prompt_template,
        )
        if result is None:
            return self._empty_criteria_row()

        return {c: result.get(c) for c in self.criteria}

    def run(self) -> None:
        for model_name in self.answer_models:
            final_col = self._model_to_col(model_name, "final")

            if final_col not in self.df.columns:
                logger.warning(f"Column {final_col} not found, skipping {model_name}")
                continue

            logger.info(f"Evaluating: {model_name}")
            rows = []

            for _, row in tqdm(self.df.iterrows(), total=len(self.df), desc=model_name):
                criteria_scores = self._evaluate_row(
                    question=row["question"],
                    answer=row[final_col],
                )
                rows.append(criteria_scores)
                time.sleep(0.5)

            scores_df = pd.DataFrame(rows)
            result_df = pd.concat(
                [
                    self.df[["question", "answer", final_col]].rename(
                        columns={final_col: "model_answer", "answer": "reference_answer"}
                    ).reset_index(drop=True),
                    scores_df.reset_index(drop=True),
                ],
                axis=1,
            )
            self.results[model_name] = result_df
            self._save(model_name, result_df)

    def _save(self, model_name: str, df: pd.DataFrame) -> None:
        filename = f"judge_{model_name.replace('/', '_')}.csv"
        path = self.output_dir / filename
        df.to_csv(path, index=False, sep=";")
        logger.info(f"Saved: {path}")

    def print_summary(self) -> None:
        for model_name, df in self.results.items():
            means = df[self.criteria].mean().round(2)
            total = means.sum().round(2)
            logger.info(f"{model_name}: {means.to_dict()} | total={total}")

In [ ]:
output_dir = Path(OUTPUT_DIR)
results_path = output_dir / RESULTS_FILE

judge_client = JudgeClient(
    api_url=API_URL,
    api_key=API_KEY,
    model=JUDGE_MODEL,
    max_retries=MAX_RETRIES,
    timeout=TIMEOUT,
    retry_delay=RETRY_DELAY,
)

judge_pipeline = JudgePipeline(
    results_path=results_path,
    output_dir=output_dir,
    judge_client=judge_client,
    prompt_template=JUDGE_CRITERIA,
    answer_models=ANSWER_MODELS,
    criteria=CRITERIA,
    eval_rows=EVAL_ROWS,
)

judge_pipeline.load_data()
judge_pipeline.run()
judge_pipeline.print_summary()

In [ ]:
for model_name, df in judge_pipeline.results.items():
    print(f"\n{model_name}")
    pd.set_option("display.max_colwidth", 200)
    display(df)

# Обработка результатов LLM-as-a-judge и человеческой оценки

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/')

In [ ]:
import logging
import sys
from pathlib import Path

import pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import cohen_kappa_score

OUTPUT_DIR = "/content/gdrive/My Drive/Agile/"

In [ ]:
MODEL_FILES = {
    "gemini-3-pro-preview":       "judge_gemini-3-pro-preview.xlsx",
    "claude-sonnet-4.6":          "judge_anthropic_claude-sonnet-4.6.xlsx",
    "avibe":                       "judge_AvitoTech_avibe.xlsx",
    "t-lite-it-1.0":              "judge_t-tech_T-lite-it-1.0.xlsx",
}

CRITERIA = [
    "agile_correctness",
    "practicality",
    "context_handling",
    "politeness",
    "text_cleanliness",
    "language_quality",
]

WEIGHTS = {
    "agile_correctness":  0.1,
    "practicality":       0.3,
    "context_handling":   0.1,
    "politeness":         0.1,
    "text_cleanliness":   0.2,
    "language_quality":   0.2,
}

In [ ]:
root_logger = logging.getLogger()
root_logger.handlers.clear()
root_logger.setLevel(logging.INFO)
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
root_logger.addHandler(handler)
logger = logging.getLogger(__name__)

In [ ]:
class DataLoader:
    def __init__(self, output_dir: Path, model_files: dict, criteria: list) -> None:
        self.output_dir = output_dir
        self.model_files = model_files
        self.criteria = criteria
        self.dataframes: dict[str, pd.DataFrame] = {}

    def load(self) -> None:
        for model_name, filename in self.model_files.items():
            path = self.output_dir / filename
            df = pd.read_excel(path)
            logger.info(f"Loaded {model_name}: {len(df)} rows, columns: {list(df.columns)}")
            self.dataframes[model_name] = df

    def validate(self) -> None:
        llm_cols = self.criteria
        human_cols = [f"{c}_human" for c in self.criteria]
        all_score_cols = llm_cols + human_cols
        has_errors = False

        for model_name, df in self.dataframes.items():
            missing = df[all_score_cols].isnull().sum()
            missing = missing[missing > 0]
            if not missing.empty:
                logger.warning(f"{model_name} has missing values:\n{missing.to_string()}")
                has_errors = True
            else:
                logger.info(f"{model_name}: no missing values")

        if not has_errors:
            logger.info("Validation passed: all score columns are complete")

In [ ]:
class ScoreAggregator:
    def __init__(
        self,
        dataframes: dict[str, pd.DataFrame],
        criteria: list,
        weights: dict,
    ) -> None:
        self.dataframes = dataframes
        self.criteria = criteria
        self.weights = weights
        self.summary: pd.DataFrame = None

    @staticmethod
    def _weighted_score(row: pd.Series, criteria: list, weights: dict, suffix: str) -> float:
        return sum(row[f"{c}{suffix}"] * weights[c] for c in criteria)

    def build_summary(self) -> None:
        rows = []

        for model_name, df in self.dataframes.items():
            row = {"model": model_name}

            for c in self.criteria:
                row[f"{c}_llm"]   = df[c].mean().round(3)
                row[f"{c}_human"] = df[f"{c}_human"].mean().round(3)

            row["total_llm"] = round(
                sum(row[f"{c}_llm"] * self.weights[c] for c in self.criteria), 3
            )
            row["total_human"] = round(
                sum(row[f"{c}_human"] * self.weights[c] for c in self.criteria), 3
            )

            rows.append(row)

        self.summary = pd.DataFrame(rows).set_index("model")

    def sort_summary(self) -> None:
        self.summary = self.summary.sort_values("total_human", ascending=False)

In [ ]:
class AgreementAnalyzer:
    def __init__(
        self,
        dataframes: dict[str, pd.DataFrame],
        summary: pd.DataFrame,
        criteria: list,
        weights: dict,
    ) -> None:
        self.dataframes = dataframes
        self.summary = summary
        self.criteria = criteria
        self.weights = weights
        self.per_criterion: pd.DataFrame = None
        self.per_model: pd.DataFrame = None

    def analyze_per_criterion(self) -> None:
        rows = []
        for model_name, df in self.dataframes.items():
            row = {"model": model_name}
            for c in self.criteria:
                llm_col   = df[c].astype(int)
                human_col = df[f"{c}_human"].astype(int)

                exact_match = (llm_col == human_col).mean().round(3)

                # Phi = Pearson r для бинарных переменных
                try:
                    phi, _ = pearsonr(llm_col, human_col)
                    phi = round(phi, 3)
                except Exception:
                    phi = float("nan")

                # Cohen's kappa — совпадение с поправкой на случайное угадывание
                try:
                    kappa = cohen_kappa_score(llm_col, human_col)
                    kappa = round(kappa, 3)
                except Exception:
                    kappa = float("nan")

                row[f"{c}_exact"] = exact_match
                row[f"{c}_phi"]   = phi
                row[f"{c}_kappa"] = kappa
            rows.append(row)
        self.per_criterion = pd.DataFrame(rows).set_index("model")

    def analyze_per_model(self) -> None:
        llm_ranks   = self.summary["total_llm"].rank(ascending=False).astype(int)
        human_ranks = self.summary["total_human"].rank(ascending=False).astype(int)

        self.per_model = pd.DataFrame({
            "total_llm":    self.summary["total_llm"],
            "rank_llm":     llm_ranks,
            "total_human":  self.summary["total_human"],
            "rank_human":   human_ranks,
            "rank_delta":   (llm_ranks - human_ranks).abs(),
        })

    def interpret(self) -> str:
        if self.per_model is None:
            return "Run analyze_per_model() first"

        max_delta  = self.per_model["rank_delta"].max()
        mean_delta = self.per_model["rank_delta"].mean().round(2)
        exact_rank = (self.per_model["rank_delta"] == 0).sum()
        n          = len(self.per_model)

        if mean_delta == 0:
            level = "полное совпадение ранжирования"
        elif mean_delta <= 0.5:
            level = "высокое совпадение ранжирования"
        elif mean_delta <= 1.0:
            level = "умеренное совпадение ранжирования"
        else:
            level = "слабое совпадение ранжирования"

        return (
            f"Согласованность LLM-судьи с человеком по ранжированию моделей: {level}. "
            f"Точных совпадений рангов: {exact_rank}/{n}. "
            f"Среднее отклонение ранга: {mean_delta}, максимальное: {max_delta}."
        )

In [ ]:
output_dir = Path(OUTPUT_DIR)

loader = DataLoader(
    output_dir=output_dir,
    model_files=MODEL_FILES,
    criteria=CRITERIA,
)
loader.load()
loader.validate()

aggregator = ScoreAggregator(
    dataframes=loader.dataframes,
    criteria=CRITERIA,
    weights=WEIGHTS,
)
aggregator.build_summary()
aggregator.sort_summary()

analyzer = AgreementAnalyzer(
    dataframes=loader.dataframes,
    summary=aggregator.summary,
    criteria=CRITERIA,
    weights=WEIGHTS,
)
analyzer.analyze_per_criterion()
analyzer.analyze_per_model()

In [ ]:
pd.set_option("display.float_format", "{:.3f}".format)
pd.set_option("display.max_columns", None)

print("=== Средние оценки по критериям и итоговая взвешенная оценка ===")
display(aggregator.summary)

print("\n=== Точное совпадение и корреляция Пирсона по каждому критерию ===")
display(analyzer.per_criterion)

print("\n=== Согласованность итоговых взвешенных оценок (LLM vs Human) ===")
display(analyzer.per_model)

print(f"\n{analyzer.interpret()}")

In [ ]:
for model_name, df in loader.dataframes.items():
    print(f"\n{model_name}")
    for c in CRITERIA:
        llm_std   = df[c].std()
        human_std = df[f"{c}_human"].std()
        if llm_std == 0 or human_std == 0:
            print(f"  {c}: llm_std={llm_std:.3f}, human_std={human_std:.3f} — нет вариации")

In [ ]:
output_path = Path(OUTPUT_DIR) / "llm_analysis_results.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    aggregator.summary.to_excel(writer, sheet_name="summary")
    analyzer.per_criterion.to_excel(writer, sheet_name="per_criterion")
    analyzer.per_model.to_excel(writer, sheet_name="per_model")

logger.info(f"Saved: {output_path}")

# Корректировка промпта для LLM-as-a-judge

**agile_correctness** (LLM 0.10–0.65 vs Human 0.20–0.65) — судья был слишком строг. Исправление: упрощения и обобщения в формате мессенджера теперь допустимы, 0 только при явном противоречии Agile Manifesto или вредном совете.

**context_handling** — самый большой разрыв (LLM 0.30–0.85 vs Human 0.90–1.00). Судья занижал почти всегда. Исправление: добавлена явная инструкция сначала определить, конкретный вопрос или общий, и засчитывать 1 если модель дала совет, применимый при любом контексте.

**practicality** (LLM 0.55–0.85 vs Human 0.95–1.00 у трёх моделей) — судья требовал слишком детальных шагов. Исправление: достаточно одного конкретного инструмента или следующего шага на высоком уровне.

**language_quality** (LLM 0.40–0.60 vs Human 0.60–1.00) — судья штрафовал за Agile-термины. Исправление: добавлен явный список допустимых англицизмов (actionable, roadmap, velocity и др.).

**Глобальное изменение:** правило разрешения неопределённости перевёрнуто — было "если есть сомнение — ставь 0", стало "если есть сомнение — ставь 1". Это ключевая причина систематического занижения.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/')

In [ ]:
import json
import logging
import re
import sys
import time
from getpass import getpass
from pathlib import Path
from typing import Optional

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from tqdm.auto import tqdm
from urllib3.util.retry import Retry

OUTPUT_DIR = "/content/gdrive/MyDrive/Agile/"
JUDGE_MODEL = "openai/gpt-5.2"
API_URL = "https://api.vsellm.ru/v1/chat/completions"
API_KEY = getpass("API Key: ")

MODEL_FILES = {
    "gemini-3-pro-preview":  "judge_gemini-3-pro-preview.xlsx",
    "claude-sonnet-4.6":     "judge_anthropic_claude-sonnet-4.6.xlsx",
    "avibe":                  "judge_AvitoTech_avibe.xlsx",
    "t-lite-it-1.0":         "judge_t-tech_T-lite-it-1.0.xlsx",
}

CRITERIA = [
    "agile_correctness",
    "practicality",
    "context_handling",
    "politeness",
    "text_cleanliness",
    "language_quality",
]

MAX_RETRIES = 3
TIMEOUT = 120
RETRY_DELAY = 5

root_logger = logging.getLogger()
root_logger.handlers.clear()
root_logger.setLevel(logging.INFO)
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
root_logger.addHandler(handler)
logger = logging.getLogger(__name__)

In [ ]:
JUDGE_CRITERIA_V2 = """
Ты — строгий проверяющий. Твоя задача — оценить ответ Agile коуча
по 6 критериям. Каждый критерий бинарный: 0 или 1.

### ОСНОВНЫЕ ИНСТРУКЦИИ ###

* Ты действуешь исключительно как **верификатор**, а не как советчик.
  Не исправляй ответ и не предлагай улучшений.
* Проверяй ответ **шаг за шагом** по каждому критерию.
* Для каждого нарушения **цитируй конкретный фрагмент** ответа,
  который стал основанием для оценки 0.
* Не допускай промежуточных значений: только 0 или 1.
  Если есть сомнение — ставь 1. Снижай оценку только при явном
  и конкретном нарушении, которое ты можешь процитировать.
* Оценивай только финальный ответ модели, не черновик.
* Помни, что это ответ в корпоративном мессенджере: стандарты
  строгости ниже, чем в академическом тексте или официальном документе.

---

### КРИТЕРИИ ###

**1. agile_correctness — Корректность с точки зрения Agile**

1 — совет не противоречит принципам Agile. Допускаются:
  - упрощения и обобщения, уместные для формата мессенджера
  - советы без ссылки на конкретную методологию, если они
    не противоречат ни одной из них
  - практики, применимые в нескольких методологиях одновременно

0 — только при явном нарушении, а именно:
  - совет прямо противоречит Agile Manifesto или Scrum Guide
    (например, рекомендация убрать ретроспективу "чтобы сэкономить время")
  - практика явно и однозначно приписана неверной методологии
    (например, "в Kanban обязателен спринт")
  - логическая ошибка настолько грубая, что совет вреден
    (например, "команда перегружена, значит нужно добавить ещё задач")

---

**2. practicality — Практическая применимость**

1 — после прочтения ответа понятно, что делать дальше.
  Достаточно одного из:
  - назван конкретный инструмент, встреча или артефакт
  - описан следующий шаг, пусть и на высоком уровне
  - задан уточняющий вопрос, который помогает сузить проблему

0 — только если:
  - ответ состоит исключительно из абстракций без единого
    конкретного действия или инструмента
    (пример нарушения: "улучшайте коммуникацию и взаимодействие")
  - после прочтения полностью непонятно, что делать

---

**3. context_handling — Работа с неполным контекстом**

Прежде чем оценивать, определи: был ли вопрос достаточно конкретным?

Если вопрос конкретный (указана методология, роль, конкретная проблема):
  1 — дан конкретный совет по существу
  0 — модель уклонилась от ответа без причины

Если вопрос общий или неполный (нет методологии, нет деталей):
  1 — выполнено хотя бы одно из:
    * модель задала уточняющий вопрос
    * модель явно назвала допущение ("если вы работаете по Scrum, то...")
    * модель дала совет, применимый при любом контексте
  0 — только если модель дала уверенный специфичный совет,
    явно предполагающий контекст, которого нет в вопросе,
    без каких-либо оговорок

---

**4. politeness — Вежливость и тон**

1 — тон соответствует дружелюбному профессиональному стилю:
  - если в вопросе есть приветствие — ответ начинается
    с ответного приветствия
  - нет покровительственного, холодного или фамильярного тона

0 — хотя бы одно из:
  - сотрудник поздоровался, модель проигнорировала приветствие
  - тон звучит пренебрежительно или менторски
    (пример нарушения: "Вы должны понимать, что...")
  - тон избыточно фамильярный

---

**5. text_cleanliness — Чистота текста**

1 — текст не содержит артефактов:
  - нет эмодзи
  - нет висящих символов (* ** :) в начале текста
  - нет технических артефактов генерации (повторов, обрывов,
    незакрытых скобок)

0 — хотя бы одно из:
  - присутствуют эмодзи
  - текст начинается с ** или :
  - есть явные артефакты генерации

---

**6. language_quality — Грамотность и языковая чистота**

1 — текст написан грамотно и понятно на русском языке.
  Допускаются:
  - устоявшиеся Agile-термины на английском: спринт, бэклог,
    скрам, ретроспектива, стендап, воркшоп, фасилитация,
    дейли, канбан, эпик, сторипоинты, velocity, roadmap,
    actionable, фреймворк
  - написание Agile, Scrum, Kanban, SAFe, LeSS с заглавной

0 — только при явном нарушении:
  - грубые орфографические или грамматические ошибки,
    затрудняющие понимание
  - замена стандартных русских слов английскими без необходимости
    (пример нарушения: "имплементировать" вместо "внедрить",
    "эджайл-подход" вместо "Agile-подход",
    "коммитить" вместо "брать в работу")

---

### ИНСТРУКЦИЯ ПО ВЕРИФИКАЦИИ ###

Шаг 1. Прочитай вопрос сотрудника и финальный ответ модели полностью.

Шаг 2. Для каждого критерия:
  - определи, есть ли явное нарушение
  - если есть — процитируй конкретный фрагмент
  - если нарушения нет — ставь 1

Шаг 3. Проверь согласованность: если agile_correctness = 0
  из-за вредного совета, то practicality, скорее всего, тоже 0.

Шаг 4. Сформируй итоговый JSON.

---

### ФОРМАТ ВЫВОДА ###

Верни результат строго в формате JSON без пояснений вне блока:
```json
{{
  "agile_correctness": <0 или 1>,
  "practicality": <0 или 1>,
  "context_handling": <0 или 1>,
  "politeness": <0 или 1>,
  "text_cleanliness": <0 или 1>,
  "language_quality": <0 или 1>,
  "total": <сумма баллов от 0 до 6>,
  "violations": [
    {{
      "criterion": "<название критерия>",
      "quote": "<цитата из ответа>",
      "reason": "<причина нарушения>"
    }}
  ]
}}
```

Если нарушений нет — "violations" возвращается пустым списком [].

---

### ДАННЫЕ ДЛЯ ОЦЕНКИ ###

Вопрос сотрудника:
{question}

Финальный ответ модели:
{answer}
"""

In [ ]:
class JudgeClient:
    def __init__(
        self,
        api_url: str,
        api_key: str,
        model: str,
        max_retries: int = 3,
        timeout: int = 120,
        retry_delay: int = 5,
    ) -> None:
        self.api_url = api_url
        self.api_key = api_key
        self.model = model
        self.max_retries = max_retries
        self.timeout = timeout
        self.retry_delay = retry_delay

    def _create_session(self) -> requests.Session:
        session = requests.Session()
        retry = Retry(
            total=self.max_retries,
            backoff_factor=1,
            status_forcelist=[429, 500, 502, 503, 504],
            allowed_methods=["POST"],
        )
        adapter = HTTPAdapter(max_retries=retry)
        session.mount("http://", adapter)
        session.mount("https://", adapter)
        return session

    def evaluate(self, question: str, answer: str, prompt_template: str) -> Optional[dict]:
        prompt = prompt_template.format(question=question, answer=answer)
        headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {self.api_key}",
        }
        payload = {
            "model": self.model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.0,
        }
        session = self._create_session()

        for attempt in range(self.max_retries):
            try:
                response = session.post(
                    self.api_url,
                    headers=headers,
                    json=payload,
                    timeout=self.timeout,
                )
                response.raise_for_status()
                content = (
                    response.json()
                    .get("choices", [{}])[0]
                    .get("message", {})
                    .get("content", "")
                    .strip()
                )
                return self._parse_json(content)
            except Exception as e:
                logger.warning(f"Attempt {attempt + 1} failed: {e}")
                time.sleep(self.retry_delay * (attempt + 1))

        logger.error("All attempts failed")
        return None

    @staticmethod
    def _parse_json(content: str) -> Optional[dict]:
        clean = re.sub(r"```json|```", "", content).strip()
        try:
            return json.loads(clean)
        except json.JSONDecodeError as e:
            logger.error(f"JSON parse error: {e}. Content: {content[:200]}")
            return None

In [ ]:
class RescoringPipeline:
    def __init__(
        self,
        output_dir: Path,
        model_files: dict,
        judge_client: JudgeClient,
        prompt_template: str,
        criteria: list,
    ) -> None:
        self.output_dir = output_dir
        self.model_files = model_files
        self.judge_client = judge_client
        self.prompt_template = prompt_template
        self.criteria = criteria

    def _empty_scores(self) -> dict:
        return {c: None for c in self.criteria}

    def _evaluate_row(self, question: str, answer: str) -> dict:
        if not isinstance(answer, str) or answer.startswith("[ERROR"):
            return self._empty_scores()
        result = self.judge_client.evaluate(
            question=question,
            answer=answer,
            prompt_template=self.prompt_template,
        )
        if result is None:
            return self._empty_scores()
        return {c: result.get(c) for c in self.criteria}

    def run(self) -> None:
        for model_name, filename in self.model_files.items():
            input_path = self.output_dir / filename
            df = pd.read_excel(input_path)
            logger.info(f"Loaded {model_name}: {len(df)} rows")

            human_cols = [f"{c}_human" for c in self.criteria]
            cols_to_keep = ["question", "reference_answer", "model_answer"] + human_cols
            df_out = df[cols_to_keep].copy()

            new_scores = []
            for _, row in tqdm(df_out.iterrows(), total=len(df_out), desc=model_name):
                scores = self._evaluate_row(
                    question=row["question"],
                    answer=row["model_answer"],
                )
                new_scores.append(scores)
                time.sleep(0.5)

            scores_df = pd.DataFrame(new_scores)
            for c in self.criteria:
                df_out.insert(
                    df_out.columns.get_loc(f"{c}_human"),
                    c,
                    scores_df[c],
                )

            output_name = filename.replace(".xlsx", "_v2.xlsx")
            output_path = self.output_dir / output_name
            df_out.to_excel(output_path, index=False)
            logger.info(f"Saved: {output_path}")

In [ ]:
output_dir = Path(OUTPUT_DIR)

judge_client = JudgeClient(
    api_url=API_URL,
    api_key=API_KEY,
    model=JUDGE_MODEL,
    max_retries=MAX_RETRIES,
    timeout=TIMEOUT,
    retry_delay=RETRY_DELAY,
)

pipeline = RescoringPipeline(
    output_dir=output_dir,
    model_files=MODEL_FILES,
    judge_client=judge_client,
    prompt_template=JUDGE_CRITERIA_V2,
    criteria=CRITERIA,
)

pipeline.run()

In [ ]:
WEIGHTS = {
    "agile_correctness":  0.1,
    "practicality":       0.3,
    "context_handling":   0.1,
    "politeness":         0.1,
    "text_cleanliness":   0.2,
    "language_quality":   0.2,
}

In [ ]:
MODEL_FILES = {
    "gemini-3-pro-preview":  "judge_gemini-3-pro-preview_v2.xlsx",
    "claude-sonnet-4.6":     "judge_anthropic_claude-sonnet-4.6_v2.xlsx",
    "avibe":                  "judge_AvitoTech_avibe_v2.xlsx",
    "t-lite-it-1.0":         "judge_t-tech_T-lite-it-1.0_v2.xlsx",
}

In [ ]:
OUTPUT_DIR = "/content/gdrive/My Drive/Agile/"

In [ ]:
output_dir = Path(OUTPUT_DIR)

loader = DataLoader(
    output_dir=output_dir,
    model_files=MODEL_FILES,
    criteria=CRITERIA,
)
loader.load()
loader.validate()

aggregator = ScoreAggregator(
    dataframes=loader.dataframes,
    criteria=CRITERIA,
    weights=WEIGHTS,
)
aggregator.build_summary()
aggregator.sort_summary()

analyzer = AgreementAnalyzer(
    dataframes=loader.dataframes,
    summary=aggregator.summary,
    criteria=CRITERIA,
    weights=WEIGHTS,
)
analyzer.analyze_per_criterion()
analyzer.analyze_per_model()

In [ ]:
pd.set_option("display.float_format", "{:.3f}".format)
pd.set_option("display.max_columns", None)

print("=== Средние оценки по критериям и итоговая взвешенная оценка ===")
display(aggregator.summary)

print("\n=== Точное совпадение и корреляция Пирсона по каждому критерию ===")
display(analyzer.per_criterion)

print("\n=== Согласованность итоговых взвешенных оценок (LLM vs Human) ===")
display(analyzer.per_model)

print(f"\n{analyzer.interpret()}")

In [ ]:
for model_name, df in loader.dataframes.items():
    print(f"\n{model_name}")
    for c in CRITERIA:
        llm_std   = df[c].std()
        human_std = df[f"{c}_human"].std()
        if llm_std == 0 or human_std == 0:
            print(f"  {c}: llm_std={llm_std:.3f}, human_std={human_std:.3f} — нет вариации")

In [ ]:
output_path = Path(OUTPUT_DIR) / "llm_analysis_results_v2.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    aggregator.summary.to_excel(writer, sheet_name="summary")
    analyzer.per_criterion.to_excel(writer, sheet_name="per_criterion")
    analyzer.per_model.to_excel(writer, sheet_name="per_model")

logger.info(f"Saved: {output_path}")

### Повторные эксперименты с версионированием

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/')

import json
import logging
import re
import sys
import time
from getpass import getpass
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from sklearn.metrics import cohen_kappa_score, confusion_matrix
from scipy.stats import pearsonr
from tqdm.auto import tqdm
from urllib3.util.retry import Retry

root_logger = logging.getLogger()
root_logger.handlers.clear()
root_logger.setLevel(logging.INFO)
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
root_logger.addHandler(handler)
logger = logging.getLogger(__name__)

OUTPUT_DIR = "/content/gdrive/MyDrive/Agile/"
JUDGE_MODEL = "openai/gpt-5.2"
API_URL = "https://api.vsellm.ru/v1/chat/completions"
API_KEY = getpass("API Key: ")

BASE_MODEL_FILES = {
    "gemini-3-pro-preview": "judge_gemini-3-pro-preview",
    "claude-sonnet-4.6":    "judge_anthropic_claude-sonnet-4.6",
    "avibe":                "judge_AvitoTech_avibe",
    "t-lite-it-1.0":        "judge_t-tech_T-lite-it-1.0",
}

CRITERIA = [
    "agile_correctness",
    "practicality",
    "context_handling",
    "politeness",
    "text_cleanliness",
    "language_quality",
]

WEIGHTS = {
    "agile_correctness": 0.1,
    "practicality":      0.3,
    "context_handling":  0.1,
    "politeness":        0.1,
    "text_cleanliness":  0.2,
    "language_quality":  0.2,
}

MAX_RETRIES = 3
TIMEOUT = 120
RETRY_DELAY = 5

In [ ]:
# Единственное место, где меняется версия
VERSION = "v9"

In [ ]:
JUDGE_PROMPTS = {}

JUDGE_PROMPTS["v1"] = """
Ты — строгий проверяющий. Твоя задача — оценить ответ Agile коуча
по 6 критериям. Каждый критерий бинарный: 0 или 1.

### ОСНОВНЫЕ ИНСТРУКЦИИ ###

* Ты действуешь исключительно как **верификатор**, а не как советчик.
  Не исправляй ответ и не предлагай улучшений.
* Проверяй ответ **шаг за шагом** по каждому критерию.
* Для каждого нарушения **цитируй конкретный фрагмент** ответа,
  который стал основанием для оценки 0.
* Не допускай промежуточных значений: только 0 или 1.
  Если есть сомнение — ставь 0.

---

### КРИТЕРИИ ###

**1. agile_correctness — Корректность с точки зрения Agile**

1 — совет полностью соответствует практикам и принципам Agile:
  - каждая упомянутая практика приписана правильной методологии
  - нет утверждений, противоречащих Agile Manifesto или Scrum Guide
  - нет логических ошибок вида "X сломано, значит делай Y"
    без причинно-следственного обоснования

0 — хотя бы одно из:
  - практика приписана неверной методологии
  - совет прямо противоречит базовым принципам Agile
  - причинно-следственная связь между проблемой и рекомендацией
    отсутствует или ложная

---

**2. practicality — Практическая применимость**

1 — ответ содержит конкретное действие, выполнимое в реальной
    корпоративной команде без дополнительных ресурсов:
  - есть понятный следующий шаг
  - рекомендация не требует идеальных условий

0 — хотя бы одно из:
  - совет абстрактный, без конкретных шагов
    (пример нарушения: "улучшите коммуникацию в команде")
  - рекомендация применима только в идеальных условиях
  - непонятно, что именно нужно сделать после прочтения

---

**3. context_handling — Работа с неполным контекстом**

1 — модель корректно обработала уровень полноты вопроса:
  - если контекст достаточен: дан конкретный совет
  - если контекст недостаточен: модель явно обозначила,
    чего не хватает, и либо задала уточняющий вопрос,
    либо дала совет для наиболее вероятного сценария
    с явным указанием допущения

0 — хотя бы одно из:
  - модель сделала неоправданные допущения без их обозначения
  - при явно неполном вопросе дан уверенный совет без оговорок
  - при достаточном контексте модель уклонилась от конкретики

---

**4. politeness — Вежливость и тон**

1 — тон соответствует дружелюбному профессиональному стилю:
  - если в вопросе есть приветствие — ответ начинается
    с ответного приветствия
  - нет покровительственного, холодного или фамильярного тона

0 — хотя бы одно из:
  - сотрудник поздоровался, модель проигнорировала приветствие
  - тон звучит пренебрежительно или менторски
    (пример нарушения: "Вы должны понимать, что...")
  - тон избыточно фамильярный

---

**5. text_cleanliness — Чистота текста**

1 — текст не содержит артефактов:
  - нет эмодзи
  - нет висящих символов (* ** :) в начале текста
  - нет технических артефактов генерации (повторов, обрывов,
    незакрытых скобок)

0 — хотя бы одно из:
  - присутствуют эмодзи
  - текст начинается с ** или :
  - есть артефакты генерации

---

**6. language_quality — Грамотность и языковая чистота**

1 — текст написан грамотно на русском языке:
  - нет орфографических и грубых пунктуационных ошибок
  - англицизмы используются только там, где нет устоявшегося
    русского эквивалента (спринт, бэклог, скрам — допустимо;
    "имплементировать" вместо "внедрить" — нарушение)

0 — хотя бы одно из:
  - есть орфографические или грубые грамматические ошибки
  - текст перегружен английскими словами без необходимости
  - используются слова там, где они неуместны
    (пример нарушения: "эджайл-подход" вместо "Agile-подход")

---

### ИНСТРУКЦИЯ ПО ВЕРИФИКАЦИИ ###

Перед выводом результата выполни следующие шаги:

Шаг 1. Прочитай вопрос сотрудника и финальный ответ модели полностью.

Шаг 2. Для каждого критерия:
  - процитируй фрагмент ответа, на который опираешься
  - зафикси: выполнен критерий или нарушен
  - если нарушен — укажи конкретную причину

Шаг 3. Сформируй итоговый JSON.

---

### ФОРМАТ ВЫВОДА ###

Верни результат строго в формате JSON без пояснений вне блока:
```json
{{{{
  "agile_correctness": <0 или 1>,
  "practicality": <0 или 1>,
  "context_handling": <0 или 1>,
  "politeness": <0 или 1>,
  "text_cleanliness": <0 или 1>,
  "language_quality": <0 или 1>,
  "total": <сумма баллов от 0 до 6>,
  "violations": [
    {{{{
      "criterion": "<название критерия>",
      "quote": "<цитата из ответа>",
      "reason": "<причина нарушения>"
    }}}}
  ]
}}}}
```

Если нарушений нет — "violations" возвращается пустым списком [].

---

### ДАННЫЕ ДЛЯ ОЦЕНКИ ###

Вопрос сотрудника:
{question}

Финальный ответ модели:
{answer}
"""


JUDGE_PROMPTS["v2"] = """
Ты — строгий проверяющий. Твоя задача — оценить ответ Agile-коуча
по 6 критериям. Каждый критерий бинарный: 0 или 1.

### ОСНОВНЫЕ ИНСТРУКЦИИ ###

* Ты действуешь исключительно как **верификатор**, а не как советчик.
  Не исправляй ответ и не предлагай улучшений.
* Проверяй ответ **шаг за шагом** по каждому критерию.
* Для каждого нарушения **цитируй конкретный фрагмент** ответа,
  который стал основанием для оценки 0.
* Не допускай промежуточных значений: только 0 или 1.
  Если есть сомнение — ставь 0.

---

### КРИТЕРИИ ###

**1. agile_correctness — Корректность с точки зрения Agile**

1 — совет в целом соответствует практикам и принципам Agile:
  - упомянутые практики не противоречат Agile Manifesto
    или Scrum Guide
  - нет грубых фактических ошибок (например, приписывание
    практики неверной методологии)
  - рекомендации логически связаны с описанной проблемой

0 — хотя бы одно из:
  - практика приписана неверной методологии
    (например, канбан-практика названа скрам-практикой)
  - совет прямо противоречит базовым принципам Agile
  - рекомендация логически не следует из описанной проблемы

Важно: незначительные неточности в терминологии при верной сути
НЕ являются основанием для 0.

---

**2. practicality — Практическая применимость**

1 — ответ содержит хотя бы одно конкретное действие,
    выполнимое в реальной команде:
  - понятно, что именно нужно сделать
  - рекомендация реалистична для типичной корпоративной среды

0 — хотя бы одно из:
  - совет полностью абстрактный, без единого конкретного шага
    (пример нарушения: «улучшите коммуникацию в команде»
    без указания способа)
  - рекомендация нереалистична или требует идеальных условий
  - совет предметно неверен (agile_correctness = 0)
    И при этом конкретные шаги основаны на этой ошибке

---

**3. context_handling — Работа с контекстом вопроса**

1 — модель дала ответ, адекватный объёму информации в вопросе:
  - при достаточном контексте: дан конкретный совет
  - при неполном контексте: допустимо ЛЮБОЕ из:
    а) задан уточняющий вопрос
    б) дан совет с явным указанием допущения
    в) дан разумный совет для наиболее типичного сценария
       (допущение может быть неявным, если сценарий очевиден)

0 — хотя бы одно из:
  - модель сделала нетипичное или экзотическое допущение
    без его обозначения
  - при достаточном контексте модель уклонилась от ответа
    и ушла в общие фразы
  - ответ не соотносится с тем, что спрашивал сотрудник

Важно: если вопрос допускает несколько разумных трактовок
и модель ответила в рамках одной из них — это НЕ нарушение.
Не требуй от модели перечисления всех возможных сценариев.

---

**4. politeness — Вежливость и тон**

1 — тон соответствует дружелюбному профессиональному стилю:
  - если в вопросе есть приветствие — ответ содержит
    ответное приветствие
  - нет покровительственного, холодного или фамильярного тона

0 — хотя бы одно из:
  - сотрудник поздоровался, модель проигнорировала приветствие
  - тон звучит пренебрежительно или менторски
    (пример нарушения: «Вы должны понимать, что...»)
  - тон избыточно фамильярный

---

**5. text_cleanliness — Чистота текста**

1 — текст не содержит артефактов:
  - нет эмодзи
  - нет висящих символов (* ** :) в начале текста
  - нет технических артефактов генерации (повторов, обрывов,
    незакрытых скобок)

0 — хотя бы одно из:
  - присутствуют эмодзи
  - текст начинается с ** или :
  - есть артефакты генерации

---

**6. language_quality — Грамотность и языковая чистота**

1 — текст написан грамотно на русском языке:
  - нет орфографических и грубых грамматических ошибок
  - профессиональный Agile-жаргон ДОПУСТИМ:
    спринт, бэклог, скрам, стендап, дейли, ретро,
    ревью, фидбэк, митинг, тимлид, продакт-оунер,
    скрам-мастер, стейкхолдер, таск, борд, эстимейт
  - англицизмы недопустимы только когда есть очевидный
    и общеупотребительный русский эквивалент
    (пример нарушения: «имплементировать» вместо «внедрить»,
    «эджайл» вместо «Agile»)

0 — хотя бы одно из:
  - есть орфографические или грубые грамматические ошибки
  - текст перегружен англицизмами сверх профессионального
    жаргона (пример нарушения: «Давайте зашедулим митинг
    для дискаша нашего перформанса»)

Важно: одиночный допустимый англицизм НЕ является основанием
для 0. Оценивай общую читаемость текста, а не отдельные слова.

---

### ИНСТРУКЦИЯ ПО ВЕРИФИКАЦИИ ###

Перед выводом результата выполни следующие шаги:

Шаг 1. Прочитай вопрос сотрудника и финальный ответ модели полностью.

Шаг 2. Для каждого критерия:
  - процитируй фрагмент ответа, на который опираешься
  - зафикси: выполнен критерий или нарушен
  - если нарушен — укажи конкретную причину

Шаг 3. Оцени каждый критерий НЕЗАВИСИМО от остальных.
  Каждая оценка должна быть обоснована собственной цитатой.

Шаг 4. Сформируй итоговый JSON.

---

### ФОРМАТ ВЫВОДА ###

Верни результат строго в формате JSON без пояснений вне блока:
```json
{{
  "agile_correctness": <0 или 1>,
  "practicality": <0 или 1>,
  "context_handling": <0 или 1>,
  "politeness": <0 или 1>,
  "text_cleanliness": <0 или 1>,
  "language_quality": <0 или 1>,
  "total": <сумма баллов от 0 до 6>,
  "violations": [
    {{
      "criterion": "<название критерия>",
      "quote": "<цитата из ответа>",
      "reason": "<причина нарушения>"
    }}
  ]
}}
```

Если нарушений нет — "violations" возвращается пустым списком [].

---

### ДАННЫЕ ДЛЯ ОЦЕНКИ ###

Вопрос сотрудника:
{question}

Финальный ответ модели:
{answer}
"""



JUDGE_PROMPTS["v3"] = """
Ты — строгий проверяющий. Твоя задача — оценить ответ Agile-коуча
по 6 критериям. Каждый критерий бинарный: 0 или 1.

### ОСНОВНЫЕ ИНСТРУКЦИИ ###

* Ты действуешь исключительно как **верификатор**, а не как советчик.
  Не исправляй ответ и не предлагай улучшений.
* Проверяй ответ **шаг за шагом** по каждому критерию.
* Для каждого нарушения **цитируй конкретный фрагмент** ответа,
  который стал основанием для оценки 0.
* Не допускай промежуточных значений: только 0 или 1.
  Если есть сомнение — ставь 0.

---

### КРИТЕРИИ ###

**1. agile_correctness — Корректность с точки зрения Agile**

1 — совет соответствует практикам и принципам Agile:
  - упомянутые практики не противоречат Agile Manifesto
    или Scrum Guide
  - нет фактических ошибок: практики приписаны верным
    методологиям, роли описаны корректно
  - рекомендации логически связаны с описанной проблемой

0 — хотя бы одно из:
  - практика приписана неверной методологии
    (например, канбан-практика названа скрам-практикой)
  - совет противоречит принципам Agile
  - описание ролей или церемоний содержит фактическую ошибку
    (например, неверно описаны обязанности скрам-мастера)
  - рекомендация логически не следует из описанной проблемы

Калибровка: оценивай содержательную корректность, а не стиль
изложения. Вопрос не в том, насколько «академично» звучит ответ,
а в том, верны ли утверждения по существу.
Если ответ содержит хотя бы одно фактически неверное
утверждение об Agile — ставь 0, даже если остальное верно.

---

**2. practicality — Практическая применимость**

1 — ответ содержит хотя бы одно конкретное действие,
    которое АДРЕСНО решает описанную проблему:
  - действие привязано к ситуации из вопроса, а не является
    универсальным советом, применимым к любой команде
  - понятно, что именно нужно сделать
  - рекомендация реалистична для типичной корпоративной среды

0 — хотя бы одно из:
  - совет полностью абстрактный, без единого конкретного шага
    (пример нарушения: «улучшите коммуникацию в команде»)
  - рекомендации шаблонные: набор общих Agile-советов,
    не привязанных к конкретной проблеме из вопроса
    (пример нарушения: на вопрос о конфликте в команде
    перечислены стандартные церемонии скрама)
  - рекомендация нереалистична или требует идеальных условий

---

**3. context_handling — Работа с контекстом вопроса**

1 — модель дала ответ, адекватный объёму информации в вопросе:
  - при достаточном контексте: дан конкретный совет
  - при неполном контексте: допустимо ЛЮБОЕ из:
    а) задан уточняющий вопрос
    б) дан совет с явным указанием допущения
    в) дан разумный совет для наиболее типичного сценария
       (допущение может быть неявным, если сценарий очевиден)

0 — хотя бы одно из:
  - модель сделала нетипичное или экзотическое допущение
    без его обозначения
  - при достаточном контексте модель уклонилась от ответа
    и ушла в общие фразы
  - ответ не соотносится с тем, что спрашивал сотрудник

Важно: если вопрос допускает несколько разумных трактовок
и модель ответила в рамках одной из них — это НЕ нарушение.
Не требуй от модели перечисления всех возможных сценариев.

---

**4. politeness — Вежливость и тон**

1 — тон соответствует дружелюбному профессиональному стилю:
  - если в вопросе есть приветствие — ответ содержит
    ответное приветствие
  - нет покровительственного, холодного или фамильярного тона

0 — хотя бы одно из:
  - сотрудник поздоровался, модель проигнорировала приветствие
  - тон звучит пренебрежительно или менторски
    (пример нарушения: «Вы должны понимать, что...»)
  - тон избыточно фамильярный

---

**5. text_cleanliness — Чистота текста**

1 — текст не содержит артефактов:
  - нет эмодзи
  - нет висящих символов (* ** :) в начале текста
  - нет технических артефактов генерации (повторов, обрывов,
    незакрытых скобок)

0 — хотя бы одно из:
  - присутствуют эмодзи
  - текст начинается с ** или :
  - есть артефакты генерации

---

**6. language_quality — Грамотность и языковая чистота**

1 — текст написан грамотно на русском языке:
  - нет орфографических и грубых грамматических ошибок
  - устоявшийся профессиональный Agile-жаргон допустим:
    спринт, бэклог, скрам, стендап, дейли, ретро,
    ревью, фидбэк, митинг, тимлид, продакт-оунер,
    скрам-мастер, стейкхолдер, таск, борд, эстимейт

0 — хотя бы одно из:
  - есть орфографические или грубые грамматические ошибки
  - англицизмы используются там, где есть общеупотребительный
    русский эквивалент (пример нарушения: «имплементировать»
    вместо «внедрить», «эджайл» вместо «Agile»)
  - текст перегружен англицизмами сверх профессионального
    жаргона (пример нарушения: «зашедулим митинг для дискаша
    нашего перформанса»)

Калибровка: оценивай общую читаемость. Одно-два слова
из допустимого списка — норма. Но если англицизмы
выходят за пределы списка и при этом имеют русские аналоги —
это основание для 0.

---

### ИНСТРУКЦИЯ ПО ВЕРИФИКАЦИИ ###

Перед выводом результата выполни следующие шаги:

Шаг 1. Прочитай вопрос сотрудника и финальный ответ модели полностью.

Шаг 2. Для каждого критерия:
  - процитируй фрагмент ответа, на который опираешься
  - зафикси: выполнен критерий или нарушен
  - если нарушен — укажи конкретную причину

Шаг 3. Оцени каждый критерий НЕЗАВИСИМО от остальных.
  Каждая оценка должна быть обоснована собственной цитатой.

Шаг 4. Сформируй итоговый JSON.

---

### ФОРМАТ ВЫВОДА ###

Верни результат строго в формате JSON без пояснений вне блока:
```json
{{
  "agile_correctness": <0 или 1>,
  "practicality": <0 или 1>,
  "context_handling": <0 или 1>,
  "politeness": <0 или 1>,
  "text_cleanliness": <0 или 1>,
  "language_quality": <0 или 1>,
  "total": <сумма баллов от 0 до 6>,
  "violations": [
    {{
      "criterion": "<название критерия>",
      "quote": "<цитата из ответа>",
      "reason": "<причина нарушения>"
    }}
  ]
}}
```

Если нарушений нет — "violations" возвращается пустым списком [].

---

### ДАННЫЕ ДЛЯ ОЦЕНКИ ###

Вопрос сотрудника:
{question}

Финальный ответ модели:
{answer}
"""

JUDGE_PROMPTS["v4"] = """
Ты — строгий проверяющий. Твоя задача — оценить ответ Agile-коуча
по 6 критериям. Каждый критерий бинарный: 0 или 1.

### ОСНОВНЫЕ ИНСТРУКЦИИ ###

* Ты действуешь исключительно как **верификатор**, а не как советчик.
  Не исправляй ответ и не предлагай улучшений.
* Проверяй ответ **шаг за шагом** по каждому критерию.
* Для каждого нарушения **цитируй конкретный фрагмент** ответа,
  который стал основанием для оценки 0.
* Не допускай промежуточных значений: только 0 или 1.
  Если есть сомнение — ставь 0.

---

### КРИТЕРИИ ###

**1. agile_correctness — Корректность с точки зрения Agile**

1 — совет соответствует практикам и принципам Agile:
  - упомянутые практики не противоречат Agile Manifesto
    или Scrum Guide
  - нет фактических ошибок: практики приписаны верным
    методологиям, роли описаны корректно
  - рекомендации логически связаны с описанной проблемой

0 — хотя бы одно из:
  - практика приписана неверной методологии
  - совет противоречит принципам Agile
  - описание ролей или церемоний содержит фактическую ошибку
  - рекомендация логически не следует из описанной проблемы

Примеры фактических ошибок (→ оценка 0):
  - «скрам-мастер назначает задачи разработчикам»
    (нарушение: команда самоорганизующаяся)
  - «ретроспектива проводится в середине спринта»
    (нарушение: ретро — по завершении спринта)
  - «в канбане обязательны двухнедельные итерации»
    (нарушение: канбан не требует фиксированных итераций)
  - «Product Owner отвечает за техническую архитектуру»
    (нарушение: PO отвечает за бизнес-ценность бэклога)

Примеры допустимого (→ НЕ является ошибкой):
  - использование слова «стендап» вместо «Daily Scrum»
  - совет провести ретро, даже если команда не по скраму
    (ретро — общая Agile-практика)
  - нестрогий порядок описания церемоний
  - упрощённое, но верное по сути описание роли

---

**2. practicality — Практическая применимость**

1 — ответ содержит хотя бы одно конкретное действие,
    которое АДРЕСНО решает описанную проблему:
  - действие привязано к ситуации из вопроса, а не является
    универсальным советом, применимым к любой команде
  - понятно, что именно нужно сделать
  - рекомендация реалистична для типичной корпоративной среды

0 — хотя бы одно из:
  - совет полностью абстрактный, без единого конкретного шага
    (пример нарушения: «улучшите коммуникацию в команде»)
  - рекомендации шаблонные: набор общих Agile-советов,
    не привязанных к конкретной проблеме из вопроса
    (пример нарушения: на вопрос о конфликте в команде
    перечислены стандартные церемонии скрама)
  - рекомендация нереалистична или требует идеальных условий

---

**3. context_handling — Работа с контекстом вопроса**

1 — модель дала ответ, адекватный объёму информации в вопросе:
  - при достаточном контексте: дан конкретный совет
  - при неполном контексте: допустимо ЛЮБОЕ из:
    а) задан уточняющий вопрос
    б) дан совет с явным указанием допущения
    в) дан разумный совет для наиболее типичного сценария
       (допущение может быть неявным, если сценарий очевиден)

0 — хотя бы одно из:
  - модель сделала нетипичное или экзотическое допущение
    без его обозначения
  - при достаточном контексте модель уклонилась от ответа
    и ушла в общие фразы
  - ответ не соотносится с тем, что спрашивал сотрудник

Важно: если вопрос допускает несколько разумных трактовок
и модель ответила в рамках одной из них — это НЕ нарушение.
Не требуй от модели перечисления всех возможных сценариев.

---

**4. politeness — Вежливость и тон**

1 — тон соответствует дружелюбному профессиональному стилю:
  - если в вопросе есть приветствие — ответ содержит
    ответное приветствие
  - нет покровительственного, холодного или фамильярного тона

0 — хотя бы одно из:
  - сотрудник поздоровался, модель проигнорировала приветствие
  - тон звучит пренебрежительно или менторски
    (пример нарушения: «Вы должны понимать, что...»)
  - тон избыточно фамильярный

---

**5. text_cleanliness — Чистота текста**

1 — текст не содержит артефактов:
  - нет эмодзи
  - нет висящих символов (* ** :) в начале текста
  - нет технических артефактов генерации (повторов, обрывов,
    незакрытых скобок)

0 — хотя бы одно из:
  - присутствуют эмодзи
  - текст начинается с ** или :
  - есть артефакты генерации

---

**6. language_quality — Грамотность и языковая чистота**

1 — текст написан грамотно на русском языке:
  - нет орфографических и грубых грамматических ошибок
  - профессиональный Agile-жаргон ДОПУСТИМ:
    спринт, бэклог, скрам, стендап, дейли, ретро,
    ревью, фидбэк, митинг, тимлид, продакт-оунер,
    скрам-мастер, стейкхолдер, таск, борд, эстимейт

0 — хотя бы одно из:
  - есть орфографические или грубые грамматические ошибки
  - текст перегружен англицизмами сверх профессионального
    жаргона (пример нарушения: «Давайте зашедулим митинг
    для дискаша нашего перформанса»)

Важно: одиночный допустимый англицизм НЕ является основанием
для 0. Оценивай общую читаемость текста, а не отдельные слова.

---

### ИНСТРУКЦИЯ ПО ВЕРИФИКАЦИИ ###

Перед выводом результата выполни следующие шаги:

Шаг 1. Прочитай вопрос сотрудника и финальный ответ модели полностью.

Шаг 2. Для каждого критерия:
  - процитируй фрагмент ответа, на который опираешься
  - зафикси: выполнен критерий или нарушен
  - если нарушен — укажи конкретную причину

Шаг 3. Оцени каждый критерий НЕЗАВИСИМО от остальных.
  Каждая оценка должна быть обоснована собственной цитатой.

Шаг 4. Сформируй итоговый JSON.

---

### ФОРМАТ ВЫВОДА ###

Верни результат строго в формате JSON без пояснений вне блока:
```json
{{
  "agile_correctness": <0 или 1>,
  "practicality": <0 или 1>,
  "context_handling": <0 или 1>,
  "politeness": <0 или 1>,
  "text_cleanliness": <0 или 1>,
  "language_quality": <0 или 1>,
  "total": <сумма баллов от 0 до 6>,
  "violations": [
    {{
      "criterion": "<название критерия>",
      "quote": "<цитата из ответа>",
      "reason": "<причина нарушения>"
    }}
  ]
}}
```

Если нарушений нет — "violations" возвращается пустым списком [].

---

### ДАННЫЕ ДЛЯ ОЦЕНКИ ###

Вопрос сотрудника:
{question}

Финальный ответ модели:
{answer}
"""

JUDGE_PROMPTS["v5"] = """
Ты — строгий проверяющий. Твоя задача — оценить ответ Agile-коуча
по 6 критериям. Каждый критерий бинарный: 0 или 1.

### ОСНОВНЫЕ ИНСТРУКЦИИ ###

* Ты действуешь исключительно как **верификатор**, а не как советчик.
  Не исправляй ответ и не предлагай улучшений.
* Проверяй ответ **шаг за шагом** по каждому критерию.
* Для каждого нарушения **цитируй конкретный фрагмент** ответа,
  который стал основанием для оценки 0.
* Не допускай промежуточных значений: только 0 или 1.
  Если есть сомнение — ставь 0.

---

### КРИТЕРИИ ###

**1. agile_correctness — Корректность с точки зрения Agile**

1 — совет соответствует практикам и принципам Agile:
  - упомянутые практики не противоречат Agile Manifesto
    или Scrum Guide
  - нет фактических ошибок: практики приписаны верным
    методологиям, роли описаны корректно
  - рекомендации логически связаны с описанной проблемой

0 — хотя бы одно из:
  - практика приписана неверной методологии
  - совет противоречит принципам Agile
  - описание ролей или церемоний содержит фактическую ошибку
  - рекомендация логически не следует из описанной проблемы

Примеры фактических ошибок (→ оценка 0):
  - «скрам-мастер назначает задачи разработчикам»
    (нарушение: команда самоорганизующаяся)
  - «ретроспектива проводится в середине спринта»
    (нарушение: ретро — по завершении спринта)
  - «в канбане обязательны двухнедельные итерации»
    (нарушение: канбан не требует фиксированных итераций)
  - «Product Owner отвечает за техническую архитектуру»
    (нарушение: PO отвечает за бизнес-ценность бэклога)

Примеры допустимого (→ НЕ является ошибкой):
  - использование слова «стендап» вместо «Daily Scrum»
  - совет провести ретро, даже если команда не по скраму
    (ретро — общая Agile-практика)
  - нестрогий порядок описания церемоний
  - упрощённое, но верное по сути описание роли

---

**2. practicality — Практическая применимость**

1 — ответ содержит хотя бы одно конкретное действие,
    связанное с ситуацией из вопроса:
  - понятно, что именно нужно сделать
  - рекомендация реалистична для типичной корпоративной среды
  - действие хотя бы кратко объясняется в контексте
    описанной проблемы

0 — хотя бы одно из:
  - совет полностью абстрактный, без единого конкретного шага
    (пример нарушения: «улучшите коммуникацию в команде»)
  - ответ — перечисление стандартных Agile-практик
    без объяснения, как они решают конкретную проблему
    (пример: на вопрос о срыве сроков — список «дейли,
    ретро, планирование» без привязки к срокам)
  - рекомендация нереалистична или требует идеальных условий

---

**3. context_handling — Работа с контекстом вопроса**

1 — модель дала ответ, адекватный объёму информации в вопросе:
  - при достаточном контексте: дан конкретный совет
  - при неполном контексте: допустимо ЛЮБОЕ из:
    а) задан уточняющий вопрос
    б) дан совет с явным указанием допущения
    в) дан разумный совет для наиболее типичного сценария
       (допущение может быть неявным, если сценарий очевиден)

0 — хотя бы одно из:
  - модель сделала нетипичное или экзотическое допущение
    без его обозначения
  - при достаточном контексте модель уклонилась от ответа
    и ушла в общие фразы
  - ответ не соотносится с тем, что спрашивал сотрудник

Важно: если вопрос допускает несколько разумных трактовок
и модель ответила в рамках одной из них — это НЕ нарушение.
Не требуй от модели перечисления всех возможных сценариев.

---

**4. politeness — Вежливость и тон**

1 — тон соответствует дружелюбному профессиональному стилю:
  - если в вопросе есть приветствие — ответ содержит
    ответное приветствие
  - нет покровительственного, холодного или фамильярного тона

0 — хотя бы одно из:
  - сотрудник поздоровался, модель проигнорировала приветствие
  - тон звучит пренебрежительно или менторски
    (пример нарушения: «Вы должны понимать, что...»)
  - тон избыточно фамильярный

---

**5. text_cleanliness — Чистота текста**

1 — текст не содержит артефактов:
  - нет эмодзи
  - нет висящих символов (* ** :) в начале текста
  - нет технических артефактов генерации (повторов, обрывов,
    незакрытых скобок)

0 — хотя бы одно из:
  - присутствуют эмодзи
  - текст начинается с ** или :
  - есть артефакты генерации

---

**6. language_quality — Грамотность и языковая чистота**

1 — текст написан грамотно на русском языке:
  - нет орфографических, грамматических ошибок
    и нарушений согласования
  - профессиональный Agile-жаргон ДОПУСТИМ:
    спринт, бэклог, скрам, стендап, дейли, ретро,
    ревью, фидбэк, митинг, тимлид, продакт-оунер,
    скрам-мастер, стейкхолдер, таск, борд, эстимейт

0 — хотя бы одно из:
  - есть орфографические или грамматические ошибки
  - англицизмы вне списка при наличии русского аналога
    (примеры: «имплементировать» → «внедрить»,
    «фасилитировать» → «модерировать»,
    «коммитмент» → «обязательство», «эджайл» → «Agile»)
  - текст перегружен англицизмами (пример: «зашедулим
    митинг для дискаша нашего перформанса»)

Одиночный англицизм из списка или без русского аналога
(например, «инсайт») — НЕ нарушение.

---

### ИНСТРУКЦИЯ ПО ВЕРИФИКАЦИИ ###

Перед выводом результата выполни следующие шаги:

Шаг 1. Прочитай вопрос сотрудника и финальный ответ модели полностью.

Шаг 2. Для каждого критерия:
  - процитируй фрагмент ответа, на который опираешься
  - зафикси: выполнен критерий или нарушен
  - если нарушен — укажи конкретную причину

Шаг 3. Оцени каждый критерий НЕЗАВИСИМО от остальных.
  Каждая оценка должна быть обоснована собственной цитатой.

Шаг 4. Сформируй итоговый JSON.

---

### ФОРМАТ ВЫВОДА ###

Верни результат строго в формате JSON без пояснений вне блока:
```json
{{
  "agile_correctness": <0 или 1>,
  "practicality": <0 или 1>,
  "context_handling": <0 или 1>,
  "politeness": <0 или 1>,
  "text_cleanliness": <0 или 1>,
  "language_quality": <0 или 1>,
  "total": <сумма баллов от 0 до 6>,
  "violations": [
    {{
      "criterion": "<название критерия>",
      "quote": "<цитата из ответа>",
      "reason": "<причина нарушения>"
    }}
  ]
}}
```

Если нарушений нет — "violations" возвращается пустым списком [].

---

### ДАННЫЕ ДЛЯ ОЦЕНКИ ###

Вопрос сотрудника:
{question}

Финальный ответ модели:
{answer}
"""


JUDGE_PROMPTS["v6"] = """
Ты — строгий проверяющий. Твоя задача — оценить ответ Agile-коуча
по 6 критериям. Каждый критерий бинарный: 0 или 1.

### ОСНОВНЫЕ ИНСТРУКЦИИ ###

* Ты действуешь исключительно как **верификатор**, а не как советчик.
  Не исправляй ответ и не предлагай улучшений.
* Проверяй ответ **шаг за шагом** по каждому критерию.
* Для каждого нарушения **цитируй конкретный фрагмент** ответа,
  который стал основанием для оценки 0.
* Не допускай промежуточных значений: только 0 или 1.
  Если есть сомнение — ставь 0.

---

### КРИТЕРИИ ###

**1. agile_correctness — Корректность с точки зрения Agile**

1 — совет соответствует практикам и принципам Agile:
  - упомянутые практики не противоречат Agile Manifesto
    или Scrum Guide
  - нет фактических ошибок: практики приписаны верным
    методологиям, роли описаны корректно
  - рекомендации логически связаны с описанной проблемой

0 — хотя бы одно из:
  - практика приписана неверной методологии
  - совет противоречит принципам Agile
  - описание ролей или церемоний содержит фактическую ошибку
  - рекомендация логически не следует из описанной проблемы

Примеры фактических ошибок (→ оценка 0):
  - «скрам-мастер назначает задачи разработчикам»
    (нарушение: команда самоорганизующаяся)
  - «ретроспектива проводится в середине спринта»
    (нарушение: ретро — по завершении спринта)
  - «в канбане обязательны двухнедельные итерации»
    (нарушение: канбан не требует фиксированных итераций)
  - «Product Owner отвечает за техническую архитектуру»
    (нарушение: PO отвечает за бизнес-ценность бэклога)

Примеры допустимого (→ НЕ является ошибкой):
  - использование слова «стендап» вместо «Daily Scrum»
  - совет провести ретро, даже если команда не по скраму
    (ретро — общая Agile-практика)
  - нестрогий порядок описания церемоний
  - упрощённое, но верное по сути описание роли

---

**2. practicality — Практическая применимость**

1 — ответ содержит хотя бы одно конкретное действие,
    которое АДРЕСНО решает описанную проблему:
  - действие привязано к ситуации из вопроса, а не является
    универсальным советом, применимым к любой команде
  - понятно, что именно нужно сделать
  - рекомендация реалистична для типичной корпоративной среды
  Адресным считается действие, которое логически вытекает
  из описанной ситуации — даже если связь не проговорена
  явно, но очевидна из контекста.

0 — хотя бы одно из:
  - совет полностью абстрактный, без единого конкретного шага
    (пример нарушения: «улучшите коммуникацию в команде»)
  - рекомендации шаблонные: набор общих Agile-практик,
    которые подошли бы к любому вопросу про Agile,
    без объяснения, как именно они помогут в данной ситуации
    (пример: на вопрос о срыве сроков — список «дейли,
    ретро, планирование» без привязки к проблеме со сроками)
  - рекомендация нереалистична или требует идеальных условий

---

**3. context_handling — Работа с контекстом вопроса**

1 — модель дала ответ, адекватный объёму информации в вопросе:
  - при достаточном контексте: дан конкретный совет
  - при неполном контексте: допустимо ЛЮБОЕ из:
    а) задан уточняющий вопрос
    б) дан совет с явным указанием допущения
    в) дан разумный совет для наиболее типичного сценария
       (допущение может быть неявным, если сценарий очевиден)

0 — хотя бы одно из:
  - модель сделала нетипичное или экзотическое допущение
    без его обозначения
  - при достаточном контексте модель уклонилась от ответа
    и ушла в общие фразы
  - ответ не соотносится с тем, что спрашивал сотрудник

Важно: если вопрос допускает несколько разумных трактовок
и модель ответила в рамках одной из них — это НЕ нарушение.
Не требуй от модели перечисления всех возможных сценариев.

---

**4. politeness — Вежливость и тон**

1 — тон соответствует дружелюбному профессиональному стилю:
  - если в вопросе есть приветствие — ответ содержит
    ответное приветствие
  - нет покровительственного, холодного или фамильярного тона

0 — хотя бы одно из:
  - сотрудник поздоровался, модель проигнорировала приветствие
  - тон звучит пренебрежительно или менторски
    (пример нарушения: «Вы должны понимать, что...»)
  - тон избыточно фамильярный

---

**5. text_cleanliness — Чистота текста**

1 — текст не содержит артефактов:
  - нет эмодзи
  - нет висящих символов (* ** :) в начале текста
  - нет технических артефактов генерации (повторов, обрывов,
    незакрытых скобок)

0 — хотя бы одно из:
  - присутствуют эмодзи
  - текст начинается с ** или :
  - есть артефакты генерации

---

**6. language_quality — Грамотность и языковая чистота**

1 — текст написан грамотно на русском языке:
  - нет орфографических и грубых грамматических ошибок
  - профессиональный Agile-жаргон ДОПУСТИМ:
    спринт, бэклог, скрам, стендап, дейли, ретро,
    ревью, фидбэк, митинг, тимлид, продакт-оунер,
    скрам-мастер, стейкхолдер, таск, борд, эстимейт

0 — хотя бы одно из:
  - есть орфографические или грубые грамматические ошибки
  - два и более англицизма вне допустимого списка,
    для которых есть русские аналоги
  - текст перегружен англицизмами (пример: «зашедулим
    митинг для дискаша нашего перформанса»)

Одиночный англицизм из списка — норма.
Одиночный англицизм вне списка — оценивай в контексте:
если есть очевидный русский аналог — нарушение,
если нет (например, «инсайт») — допустимо.

---

### ИНСТРУКЦИЯ ПО ВЕРИФИКАЦИИ ###

Перед выводом результата выполни следующие шаги:

Шаг 1. Прочитай вопрос сотрудника и финальный ответ модели полностью.

Шаг 2. Для каждого критерия:
  - процитируй фрагмент ответа, на который опираешься
  - зафикси: выполнен критерий или нарушен
  - если нарушен — укажи конкретную причину

Шаг 3. Оцени каждый критерий НЕЗАВИСИМО от остальных.
  Каждая оценка должна быть обоснована собственной цитатой.

Шаг 4. Сформируй итоговый JSON.

---

### ФОРМАТ ВЫВОДА ###

Верни результат строго в формате JSON без пояснений вне блока:
```json
{{
  "agile_correctness": <0 или 1>,
  "practicality": <0 или 1>,
  "context_handling": <0 или 1>,
  "politeness": <0 или 1>,
  "text_cleanliness": <0 или 1>,
  "language_quality": <0 или 1>,
  "total": <сумма баллов от 0 до 6>,
  "violations": [
    {{
      "criterion": "<название критерия>",
      "quote": "<цитата из ответа>",
      "reason": "<причина нарушения>"
    }}
  ]
}}
```

Если нарушений нет — "violations" возвращается пустым списком [].

---

### ДАННЫЕ ДЛЯ ОЦЕНКИ ###

Вопрос сотрудника:
{question}

Финальный ответ модели:
{answer}
"""


JUDGE_PROMPTS["v7"] = """
Ты — строгий проверяющий. Твоя задача — оценить ответ Agile-коуча
по 6 критериям. Каждый критерий бинарный: 0 или 1.

### ОСНОВНЫЕ ИНСТРУКЦИИ ###

* Ты действуешь исключительно как **верификатор**, а не как советчик.
  Не исправляй ответ и не предлагай улучшений.
* Проверяй ответ **шаг за шагом** по каждому критерию.
* Для каждого нарушения **цитируй конкретный фрагмент** ответа,
  который стал основанием для оценки 0.
* Не допускай промежуточных значений: только 0 или 1.
  Если есть сомнение — ставь 0.

---

### КРИТЕРИИ ###

**1. agile_correctness — Корректность с точки зрения Agile**

1 — совет соответствует практикам и принципам Agile:
  - упомянутые практики не противоречат Agile Manifesto
    или Scrum Guide
  - нет фактических ошибок: практики приписаны верным
    методологиям, роли описаны корректно
  - рекомендации логически связаны с описанной проблемой

0 — хотя бы одно из:
  - практика приписана неверной методологии
  - совет противоречит принципам Agile
  - описание ролей или церемоний содержит фактическую ошибку
  - рекомендация логически не следует из описанной проблемы

Примеры фактических ошибок (→ оценка 0):
  - «скрам-мастер назначает задачи разработчикам»
    (нарушение: команда самоорганизующаяся)
  - «ретроспектива проводится в середине спринта»
    (нарушение: ретро — по завершении спринта)
  - «в канбане обязательны двухнедельные итерации»
    (нарушение: канбан не требует фиксированных итераций)
  - «Product Owner отвечает за техническую архитектуру»
    (нарушение: PO отвечает за бизнес-ценность бэклога)

Примеры допустимого (→ НЕ является ошибкой):
  - использование слова «стендап» вместо «Daily Scrum»
  - совет провести ретро, даже если команда не по скраму
    (ретро — общая Agile-практика)
  - нестрогий порядок описания церемоний
  - упрощённое, но верное по сути описание роли

---

**2. practicality — Практическая применимость**

1 — ответ содержит хотя бы одно конкретное действие,
    которое АДРЕСНО решает описанную проблему:
  - действие привязано к ситуации из вопроса, а не является
    универсальным советом, применимым к любой команде
  - понятно, что именно нужно сделать
  - рекомендация реалистична для типичной корпоративной среды

0 — хотя бы одно из:
  - совет полностью абстрактный, без единого конкретного шага
    (пример нарушения: «улучшите коммуникацию в команде»)
  - рекомендации шаблонные: набор общих Agile-советов,
    не привязанных к конкретной проблеме из вопроса
    (пример нарушения: на вопрос о конфликте в команде
    перечислены стандартные церемонии скрама)
  - рекомендация нереалистична или требует идеальных условий

Тест на шаблонность: если этот ответ можно дать без
изменений на любой другой вопрос про Agile — это 0.
Если ответ перестанет иметь смысл при замене вопроса — это 1.

---

**3. context_handling — Работа с контекстом вопроса**

1 — модель дала ответ, адекватный объёму информации в вопросе:
  - при достаточном контексте: дан конкретный совет
  - при неполном контексте: допустимо ЛЮБОЕ из:
    а) задан уточняющий вопрос
    б) дан совет с явным указанием допущения
    в) дан разумный совет для наиболее типичного сценария
       (допущение может быть неявным, если сценарий очевиден)

0 — хотя бы одно из:
  - модель сделала нетипичное или экзотическое допущение
    без его обозначения
  - при достаточном контексте модель уклонилась от ответа
    и ушла в общие фразы
  - ответ не соотносится с тем, что спрашивал сотрудник

Важно: если вопрос допускает несколько разумных трактовок
и модель ответила в рамках одной из них — это НЕ нарушение.
Не требуй от модели перечисления всех возможных сценариев.

---

**4. politeness — Вежливость и тон**

1 — тон соответствует дружелюбному профессиональному стилю:
  - если в вопросе есть приветствие — ответ содержит
    ответное приветствие
  - нет покровительственного, холодного или фамильярного тона

0 — хотя бы одно из:
  - сотрудник поздоровался, модель проигнорировала приветствие
  - тон звучит пренебрежительно или менторски
    (пример нарушения: «Вы должны понимать, что...»)
  - тон избыточно фамильярный

---

**5. text_cleanliness — Чистота текста**

1 — текст не содержит артефактов:
  - нет эмодзи
  - нет висящих символов (* ** :) в начале текста
  - нет технических артефактов генерации (повторов, обрывов,
    незакрытых скобок)

0 — хотя бы одно из:
  - присутствуют эмодзи
  - текст начинается с ** или :
  - есть артефакты генерации

---

**6. language_quality — Грамотность и языковая чистота**

1 — текст написан грамотно на русском языке:
  - нет орфографических и грубых грамматических ошибок
  - профессиональный Agile-жаргон ДОПУСТИМ:
    спринт, бэклог, скрам, стендап, дейли, ретро,
    ревью, фидбэк, митинг, тимлид, продакт-оунер,
    скрам-мастер, стейкхолдер, таск, борд, эстимейт

0 — хотя бы одно из:
  - есть орфографические или грубые грамматические ошибки
  - текст перегружен англицизмами сверх профессионального
    жаргона (пример нарушения: «Давайте зашедулим митинг
    для дискаша нашего перформанса»)

Важно: одиночный допустимый англицизм НЕ является основанием
для 0. Оценивай общую читаемость текста, а не отдельные слова.

---

### ИНСТРУКЦИЯ ПО ВЕРИФИКАЦИИ ###

Перед выводом результата выполни следующие шаги:

Шаг 1. Прочитай вопрос сотрудника и финальный ответ модели полностью.

Шаг 2. Для каждого критерия:
  - процитируй фрагмент ответа, на который опираешься
  - зафикси: выполнен критерий или нарушен
  - если нарушен — укажи конкретную причину

Шаг 3. Оцени каждый критерий НЕЗАВИСИМО от остальных.
  Каждая оценка должна быть обоснована собственной цитатой.

Шаг 4. Сформируй итоговый JSON.

---

### ФОРМАТ ВЫВОДА ###

Верни результат строго в формате JSON без пояснений вне блока:
```json
{{
  "agile_correctness": <0 или 1>,
  "practicality": <0 или 1>,
  "context_handling": <0 или 1>,
  "politeness": <0 или 1>,
  "text_cleanliness": <0 или 1>,
  "language_quality": <0 или 1>,
  "total": <сумма баллов от 0 до 6>,
  "violations": [
    {{
      "criterion": "<название критерия>",
      "quote": "<цитата из ответа>",
      "reason": "<причина нарушения>"
    }}
  ]
}}
```

Если нарушений нет — "violations" возвращается пустым списком [].

---

### ДАННЫЕ ДЛЯ ОЦЕНКИ ###

Вопрос сотрудника:
{question}

Финальный ответ модели:
{answer}
"""

# полная копия v4
JUDGE_PROMPTS["v8"] = """
Ты — строгий проверяющий. Твоя задача — оценить ответ Agile-коуча
по 6 критериям. Каждый критерий бинарный: 0 или 1.

### ОСНОВНЫЕ ИНСТРУКЦИИ ###

* Ты действуешь исключительно как **верификатор**, а не как советчик.
  Не исправляй ответ и не предлагай улучшений.
* Проверяй ответ **шаг за шагом** по каждому критерию.
* Для каждого нарушения **цитируй конкретный фрагмент** ответа,
  который стал основанием для оценки 0.
* Не допускай промежуточных значений: только 0 или 1.
  Если есть сомнение — ставь 0.

---

### КРИТЕРИИ ###

**1. agile_correctness — Корректность с точки зрения Agile**

1 — совет соответствует практикам и принципам Agile:
  - упомянутые практики не противоречат Agile Manifesto
    или Scrum Guide
  - нет фактических ошибок: практики приписаны верным
    методологиям, роли описаны корректно
  - рекомендации логически связаны с описанной проблемой

0 — хотя бы одно из:
  - практика приписана неверной методологии
  - совет противоречит принципам Agile
  - описание ролей или церемоний содержит фактическую ошибку
  - рекомендация логически не следует из описанной проблемы

Примеры фактических ошибок (→ оценка 0):
  - «скрам-мастер назначает задачи разработчикам»
    (нарушение: команда самоорганизующаяся)
  - «ретроспектива проводится в середине спринта»
    (нарушение: ретро — по завершении спринта)
  - «в канбане обязательны двухнедельные итерации»
    (нарушение: канбан не требует фиксированных итераций)
  - «Product Owner отвечает за техническую архитектуру»
    (нарушение: PO отвечает за бизнес-ценность бэклога)

Примеры допустимого (→ НЕ является ошибкой):
  - использование слова «стендап» вместо «Daily Scrum»
  - совет провести ретро, даже если команда не по скраму
    (ретро — общая Agile-практика)
  - нестрогий порядок описания церемоний
  - упрощённое, но верное по сути описание роли

---

**2. practicality — Практическая применимость**

1 — ответ содержит хотя бы одно конкретное действие,
    которое АДРЕСНО решает описанную проблему:
  - действие привязано к ситуации из вопроса, а не является
    универсальным советом, применимым к любой команде
  - понятно, что именно нужно сделать
  - рекомендация реалистична для типичной корпоративной среды

0 — хотя бы одно из:
  - совет полностью абстрактный, без единого конкретного шага
    (пример нарушения: «улучшите коммуникацию в команде»)
  - рекомендации шаблонные: набор общих Agile-советов,
    не привязанных к конкретной проблеме из вопроса
    (пример нарушения: на вопрос о конфликте в команде
    перечислены стандартные церемонии скрама)
  - рекомендация нереалистична или требует идеальных условий

---

**3. context_handling — Работа с контекстом вопроса**

1 — модель дала ответ, адекватный объёму информации в вопросе:
  - при достаточном контексте: дан конкретный совет
  - при неполном контексте: допустимо ЛЮБОЕ из:
    а) задан уточняющий вопрос
    б) дан совет с явным указанием допущения
    в) дан разумный совет для наиболее типичного сценария
       (допущение может быть неявным, если сценарий очевиден)

0 — хотя бы одно из:
  - модель сделала нетипичное или экзотическое допущение
    без его обозначения
  - при достаточном контексте модель уклонилась от ответа
    и ушла в общие фразы
  - ответ не соотносится с тем, что спрашивал сотрудник

Важно: если вопрос допускает несколько разумных трактовок
и модель ответила в рамках одной из них — это НЕ нарушение.
Не требуй от модели перечисления всех возможных сценариев.

---

**4. politeness — Вежливость и тон**

1 — тон соответствует дружелюбному профессиональному стилю:
  - если в вопросе есть приветствие — ответ содержит
    ответное приветствие
  - нет покровительственного, холодного или фамильярного тона

0 — хотя бы одно из:
  - сотрудник поздоровался, модель проигнорировала приветствие
  - тон звучит пренебрежительно или менторски
    (пример нарушения: «Вы должны понимать, что...»)
  - тон избыточно фамильярный

---

**5. text_cleanliness — Чистота текста**

1 — текст не содержит артефактов:
  - нет эмодзи
  - нет висящих символов (* ** :) в начале текста
  - нет технических артефактов генерации (повторов, обрывов,
    незакрытых скобок)

0 — хотя бы одно из:
  - присутствуют эмодзи
  - текст начинается с ** или :
  - есть артефакты генерации

---

**6. language_quality — Грамотность и языковая чистота**

1 — текст написан грамотно на русском языке:
  - нет орфографических и грубых грамматических ошибок
  - профессиональный Agile-жаргон ДОПУСТИМ:
    спринт, бэклог, скрам, стендап, дейли, ретро,
    ревью, фидбэк, митинг, тимлид, продакт-оунер,
    скрам-мастер, стейкхолдер, таск, борд, эстимейт

0 — хотя бы одно из:
  - есть орфографические или грубые грамматические ошибки
  - текст перегружен англицизмами сверх профессионального
    жаргона (пример нарушения: «Давайте зашедулим митинг
    для дискаша нашего перформанса»)

Важно: одиночный допустимый англицизм НЕ является основанием
для 0. Оценивай общую читаемость текста, а не отдельные слова.

---

### ИНСТРУКЦИЯ ПО ВЕРИФИКАЦИИ ###

Перед выводом результата выполни следующие шаги:

Шаг 1. Прочитай вопрос сотрудника и финальный ответ модели полностью.

Шаг 2. Для каждого критерия:
  - процитируй фрагмент ответа, на который опираешься
  - зафикси: выполнен критерий или нарушен
  - если нарушен — укажи конкретную причину

Шаг 3. Оцени каждый критерий НЕЗАВИСИМО от остальных.
  Каждая оценка должна быть обоснована собственной цитатой.

Шаг 4. Сформируй итоговый JSON.

---

### ФОРМАТ ВЫВОДА ###

Верни результат строго в формате JSON без пояснений вне блока:
```json
{{
  "agile_correctness": <0 или 1>,
  "practicality": <0 или 1>,
  "context_handling": <0 или 1>,
  "politeness": <0 или 1>,
  "text_cleanliness": <0 или 1>,
  "language_quality": <0 или 1>,
  "total": <сумма баллов от 0 до 6>,
  "violations": [
    {{
      "criterion": "<название критерия>",
      "quote": "<цитата из ответа>",
      "reason": "<причина нарушения>"
    }}
  ]
}}
```

Если нарушений нет — "violations" возвращается пустым списком [].

---

### ДАННЫЕ ДЛЯ ОЦЕНКИ ###

Вопрос сотрудника:
{question}

Финальный ответ модели:
{answer}
"""

# полная копия v7
JUDGE_PROMPTS["v9"] = """
Ты — строгий проверяющий. Твоя задача — оценить ответ Agile-коуча
по 6 критериям. Каждый критерий бинарный: 0 или 1.

### ОСНОВНЫЕ ИНСТРУКЦИИ ###

* Ты действуешь исключительно как **верификатор**, а не как советчик.
  Не исправляй ответ и не предлагай улучшений.
* Проверяй ответ **шаг за шагом** по каждому критерию.
* Для каждого нарушения **цитируй конкретный фрагмент** ответа,
  который стал основанием для оценки 0.
* Не допускай промежуточных значений: только 0 или 1.
  Если есть сомнение — ставь 0.

---

### КРИТЕРИИ ###

**1. agile_correctness — Корректность с точки зрения Agile**

1 — совет соответствует практикам и принципам Agile:
  - упомянутые практики не противоречат Agile Manifesto
    или Scrum Guide
  - нет фактических ошибок: практики приписаны верным
    методологиям, роли описаны корректно
  - рекомендации логически связаны с описанной проблемой

0 — хотя бы одно из:
  - практика приписана неверной методологии
  - совет противоречит принципам Agile
  - описание ролей или церемоний содержит фактическую ошибку
  - рекомендация логически не следует из описанной проблемы

Примеры фактических ошибок (→ оценка 0):
  - «скрам-мастер назначает задачи разработчикам»
    (нарушение: команда самоорганизующаяся)
  - «ретроспектива проводится в середине спринта»
    (нарушение: ретро — по завершении спринта)
  - «в канбане обязательны двухнедельные итерации»
    (нарушение: канбан не требует фиксированных итераций)
  - «Product Owner отвечает за техническую архитектуру»
    (нарушение: PO отвечает за бизнес-ценность бэклога)

Примеры допустимого (→ НЕ является ошибкой):
  - использование слова «стендап» вместо «Daily Scrum»
  - совет провести ретро, даже если команда не по скраму
    (ретро — общая Agile-практика)
  - нестрогий порядок описания церемоний
  - упрощённое, но верное по сути описание роли

---

**2. practicality — Практическая применимость**

1 — ответ содержит хотя бы одно конкретное действие,
    которое АДРЕСНО решает описанную проблему:
  - действие привязано к ситуации из вопроса, а не является
    универсальным советом, применимым к любой команде
  - понятно, что именно нужно сделать
  - рекомендация реалистична для типичной корпоративной среды

0 — хотя бы одно из:
  - совет полностью абстрактный, без единого конкретного шага
    (пример нарушения: «улучшите коммуникацию в команде»)
  - рекомендации шаблонные: набор общих Agile-советов,
    не привязанных к конкретной проблеме из вопроса
    (пример нарушения: на вопрос о конфликте в команде
    перечислены стандартные церемонии скрама)
  - рекомендация нереалистична или требует идеальных условий

Тест на шаблонность: если этот ответ можно дать без
изменений на любой другой вопрос про Agile — это 0.
Если ответ перестанет иметь смысл при замене вопроса — это 1.

---

**3. context_handling — Работа с контекстом вопроса**

1 — модель дала ответ, адекватный объёму информации в вопросе:
  - при достаточном контексте: дан конкретный совет
  - при неполном контексте: допустимо ЛЮБОЕ из:
    а) задан уточняющий вопрос
    б) дан совет с явным указанием допущения
    в) дан разумный совет для наиболее типичного сценария
       (допущение может быть неявным, если сценарий очевиден)

0 — хотя бы одно из:
  - модель сделала нетипичное или экзотическое допущение
    без его обозначения
  - при достаточном контексте модель уклонилась от ответа
    и ушла в общие фразы
  - ответ не соотносится с тем, что спрашивал сотрудник

Важно: если вопрос допускает несколько разумных трактовок
и модель ответила в рамках одной из них — это НЕ нарушение.
Не требуй от модели перечисления всех возможных сценариев.

---

**4. politeness — Вежливость и тон**

1 — тон соответствует дружелюбному профессиональному стилю:
  - если в вопросе есть приветствие — ответ содержит
    ответное приветствие
  - нет покровительственного, холодного или фамильярного тона

0 — хотя бы одно из:
  - сотрудник поздоровался, модель проигнорировала приветствие
  - тон звучит пренебрежительно или менторски
    (пример нарушения: «Вы должны понимать, что...»)
  - тон избыточно фамильярный

---

**5. text_cleanliness — Чистота текста**

1 — текст не содержит артефактов:
  - нет эмодзи
  - нет висящих символов (* ** :) в начале текста
  - нет технических артефактов генерации (повторов, обрывов,
    незакрытых скобок)

0 — хотя бы одно из:
  - присутствуют эмодзи
  - текст начинается с ** или :
  - есть артефакты генерации

---

**6. language_quality — Грамотность и языковая чистота**

1 — текст написан грамотно на русском языке:
  - нет орфографических и грубых грамматических ошибок
  - профессиональный Agile-жаргон ДОПУСТИМ:
    спринт, бэклог, скрам, стендап, дейли, ретро,
    ревью, фидбэк, митинг, тимлид, продакт-оунер,
    скрам-мастер, стейкхолдер, таск, борд, эстимейт

0 — хотя бы одно из:
  - есть орфографические или грубые грамматические ошибки
  - текст перегружен англицизмами сверх профессионального
    жаргона (пример нарушения: «Давайте зашедулим митинг
    для дискаша нашего перформанса»)

Важно: одиночный допустимый англицизм НЕ является основанием
для 0. Оценивай общую читаемость текста, а не отдельные слова.

---

### ИНСТРУКЦИЯ ПО ВЕРИФИКАЦИИ ###

Перед выводом результата выполни следующие шаги:

Шаг 1. Прочитай вопрос сотрудника и финальный ответ модели полностью.

Шаг 2. Для каждого критерия:
  - процитируй фрагмент ответа, на который опираешься
  - зафикси: выполнен критерий или нарушен
  - если нарушен — укажи конкретную причину

Шаг 3. Оцени каждый критерий НЕЗАВИСИМО от остальных.
  Каждая оценка должна быть обоснована собственной цитатой.

Шаг 4. Сформируй итоговый JSON.

---

### ФОРМАТ ВЫВОДА ###

Верни результат строго в формате JSON без пояснений вне блока:
```json
{{
  "agile_correctness": <0 или 1>,
  "practicality": <0 или 1>,
  "context_handling": <0 или 1>,
  "politeness": <0 или 1>,
  "text_cleanliness": <0 или 1>,
  "language_quality": <0 или 1>,
  "total": <сумма баллов от 0 до 6>,
  "violations": [
    {{
      "criterion": "<название критерия>",
      "quote": "<цитата из ответа>",
      "reason": "<причина нарушения>"
    }}
  ]
}}
```

Если нарушений нет — "violations" возвращается пустым списком [].

---

### ДАННЫЕ ДЛЯ ОЦЕНКИ ###

Вопрос сотрудника:
{question}

Финальный ответ модели:
{answer}
"""

assert VERSION in JUDGE_PROMPTS, f"VERSION='{VERSION}' не найдена. Доступны: {list(JUDGE_PROMPTS)}"
JUDGE_CRITERIA = JUDGE_PROMPTS[VERSION]
logger.info(f"Using prompt version: {VERSION}")


In [ ]:
# ---------------------------------------------------------------------------
# Минимальный порог успешных голосов для majority voting.
# При меньшем количестве успешных прогонов критерий получит None.
# ---------------------------------------------------------------------------
MIN_SUCCESSFUL_VOTES = 2


class JudgeClient:
    def __init__(
        self,
        api_url: str,
        api_key: str,
        model: str,
        max_retries: int = 3,
        timeout: int = 120,
        retry_delay: int = 5,
    ) -> None:
        self.api_url = api_url
        self.api_key = api_key
        self.model = model
        self.max_retries = max_retries
        self.timeout = timeout
        self.retry_delay = retry_delay

    def _create_session(self) -> requests.Session:
        session = requests.Session()
        retry = Retry(
            total=self.max_retries,
            backoff_factor=1,
            status_forcelist=[429, 500, 502, 503, 504],
            allowed_methods=["POST"],
        )
        adapter = HTTPAdapter(max_retries=retry)
        session.mount("http://", adapter)
        session.mount("https://", adapter)
        return session

    def evaluate(self, question: str, answer: str, prompt_template: str) -> Optional[dict]:
        prompt = prompt_template.format(question=question, answer=answer)
        headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {self.api_key}",
        }
        payload = {
            "model": self.model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.0,
        }
        session = self._create_session()
        for attempt in range(self.max_retries):
            try:
                response = session.post(
                    self.api_url, headers=headers, json=payload, timeout=self.timeout,
                )
                response.raise_for_status()
                content = (
                    response.json()
                    .get("choices", [{}])[0]
                    .get("message", {})
                    .get("content", "")
                    .strip()
                )
                return self._parse_json(content)
            except Exception as e:
                if hasattr(e, 'response') and e.response is not None:
                    logger.warning(f"Attempt {attempt + 1} failed: {e}. Body: {e.response.text[:300]}")
                else:
                    logger.warning(f"Attempt {attempt + 1} failed: {e}")
                time.sleep(self.retry_delay * (attempt + 1))
            # except Exception as e:
            #     logger.warning(f"Attempt {attempt + 1} failed: {e}")
            #     time.sleep(self.retry_delay * (attempt + 1))
        logger.error("All attempts failed")
        return None

    @staticmethod
    def _parse_json(content: str) -> Optional[dict]:
        clean = re.sub(r"```json|```", "", content).strip()
        try:
            return json.loads(clean)
        except json.JSONDecodeError as e:
            logger.error(f"JSON parse error: {e}. Content: {content[:200]}")
            return None


class RescoringPipeline:
    def __init__(
        self,
        output_dir: Path,
        base_model_files: dict,
        judge_client: JudgeClient,
        prompt_template: str,
        criteria: list,
        version: str,
        sleep_between_requests: float = 0.1,
    ) -> None:
        self.output_dir = output_dir
        self.base_model_files = base_model_files
        self.judge_client = judge_client
        self.prompt_template = prompt_template
        self.criteria = criteria
        self.version = version
        self.sleep_between_requests = sleep_between_requests

    def _input_path(self, base_name: str) -> Path:
        prev_version = f"v{int(self.version[1:]) - 1}"
        suffix = f"_{prev_version}.xlsx" if int(self.version[1:]) > 1 else ".xlsx"
        return self.output_dir / f"{base_name}{suffix}"

    def _output_path(self, base_name: str) -> Path:
        return self.output_dir / f"{base_name}_{self.version}.xlsx"

    def _empty_scores(self) -> dict:
        return {c: None for c in self.criteria}

    def _evaluate_row(self, question: str, answer: str) -> dict:
        if not isinstance(answer, str) or answer.startswith("[ERROR"):
            return self._empty_scores()
        result = self.judge_client.evaluate(
            question=question,
            answer=answer,
            prompt_template=self.prompt_template,
        )
        if result is None:
            return self._empty_scores()
        return {c: result.get(c) for c in self.criteria}

    def _evaluate_row_majority(
        self, question: str, answer: str, n_runs: int = 3,
    ) -> dict:
        votes = {c: [] for c in self.criteria}
        for run_idx in range(n_runs):
            scores = self._evaluate_row(question, answer)
            for c in self.criteria:
                if scores[c] is not None:
                    votes[c].append(scores[c])

        # --- Логирование числа успешных голосов ---
        vote_counts = {c: len(v) for c, v in votes.items()}
        min_votes = min(vote_counts.values())
        if min_votes < n_runs:
            lacking = {c: cnt for c, cnt in vote_counts.items() if cnt < n_runs}
            logger.warning(
                f"Incomplete voting ({min_votes}/{n_runs} min). "
                f"Per-criterion counts: {lacking}"
            )

        return {
            c: int(sum(v) > len(v) / 2)
            if len(v) >= MIN_SUCCESSFUL_VOTES
            else None
            for c, v in votes.items()
        }

    def run(self) -> None:
        for model_name, base_name in self.base_model_files.items():
            input_path = self._input_path(base_name)
            if not input_path.exists():
                logger.warning(f"File not found, skipping {model_name}: {input_path}")
                continue

            df = pd.read_excel(input_path)
            logger.info(f"Loaded {model_name}: {len(df)} rows from {input_path.name}")

            human_cols = [f"{c}_human" for c in self.criteria]
            cols_to_keep = ["question", "reference_answer", "model_answer"] + human_cols
            df_out = df[cols_to_keep].copy()

            new_scores = []
            for _, row in tqdm(df_out.iterrows(), total=len(df_out), desc=model_name):
                scores = self._evaluate_row_majority(
                    question=row["question"],
                    answer=row["model_answer"],
                    n_runs=3,
                )
                new_scores.append(scores)
                time.sleep(self.sleep_between_requests)

            scores_df = pd.DataFrame(new_scores)
            for c in self.criteria:
                df_out.insert(df_out.columns.get_loc(f"{c}_human"), c, scores_df[c])

            output_path = self._output_path(base_name)
            df_out.to_excel(output_path, index=False)
            logger.info(f"Saved: {output_path.name}")


class DataLoader:
    def __init__(self, output_dir: Path, base_model_files: dict, version: str, criteria: list) -> None:
        self.output_dir = output_dir
        self.base_model_files = base_model_files
        self.version = version
        self.criteria = criteria
        self.dataframes: dict[str, pd.DataFrame] = {}

    def load(self) -> None:
        for model_name, base_name in self.base_model_files.items():
            path = self.output_dir / f"{base_name}_{self.version}.xlsx"
            df = pd.read_excel(path)
            logger.info(f"Loaded {model_name}: {len(df)} rows")
            self.dataframes[model_name] = df

    def validate(self) -> None:
        all_score_cols = self.criteria + [f"{c}_human" for c in self.criteria]
        for model_name, df in self.dataframes.items():
            missing = df[all_score_cols].isnull().sum()
            missing = missing[missing > 0]
            if not missing.empty:
                logger.warning(f"{model_name} missing values:\n{missing.to_string()}")
            else:
                logger.info(f"{model_name}: no missing values")


class ScoreAggregator:
    def __init__(self, dataframes: dict, criteria: list, weights: dict) -> None:
        self.dataframes = dataframes
        self.criteria = criteria
        self.weights = weights
        self.summary: Optional[pd.DataFrame] = None

    def build_summary(self) -> None:
        rows = []
        for model_name, df in self.dataframes.items():
            row = {"model": model_name}
            for c in self.criteria:
                row[f"{c}_llm"]   = df[c].mean().round(3)
                row[f"{c}_human"] = df[f"{c}_human"].mean().round(3)
            row["total_llm"]   = round(sum(row[f"{c}_llm"]   * self.weights[c] for c in self.criteria), 3)
            row["total_human"] = round(sum(row[f"{c}_human"] * self.weights[c] for c in self.criteria), 3)
            rows.append(row)
        self.summary = pd.DataFrame(rows).set_index("model")

    def sort_summary(self) -> None:
        self.summary = self.summary.sort_values("total_human", ascending=False)


class AgreementAnalyzer:
    def __init__(self, dataframes: dict, summary: pd.DataFrame, criteria: list, weights: dict) -> None:
        self.dataframes = dataframes
        self.summary = summary
        self.criteria = criteria
        self.weights = weights
        self.per_criterion: Optional[pd.DataFrame] = None
        self.per_model: Optional[pd.DataFrame] = None
        self.confusion_matrices: Optional[pd.DataFrame] = None

    # ------------------------------------------------------------------
    # 1. Покритериальный анализ: exact agreement — основная метрика,
    #    phi и kappa — дополнительные (с пометкой о малой выборке).
    # ------------------------------------------------------------------
    def analyze_per_criterion(self) -> None:
        rows = []
        for model_name, df in self.dataframes.items():
            row = {"model": model_name}
            for c in self.criteria:
                llm_col   = df[c].astype(int)
                human_col = df[f"{c}_human"].astype(int)

                # --- Основная метрика ---
                row[f"{c}_exact"] = (llm_col == human_col).mean().round(3)

                # --- Дополнительные (могут быть NaN при вырожденных распределениях) ---
                try:
                    phi, _ = pearsonr(llm_col, human_col)
                    row[f"{c}_phi"] = round(phi, 3)
                except Exception:
                    row[f"{c}_phi"] = float("nan")
                try:
                    row[f"{c}_kappa"] = round(cohen_kappa_score(llm_col, human_col), 3)
                except Exception:
                    row[f"{c}_kappa"] = float("nan")

            rows.append(row)
        self.per_criterion = pd.DataFrame(rows).set_index("model")

    # ------------------------------------------------------------------
    # 2. Confusion matrix по каждому критерию, агрегированная
    #    по всем моделям. Формат: TP / FP / FN / TN
    #    (positive = оценка 1, negative = оценка 0).
    # ------------------------------------------------------------------
    def build_confusion_matrices(self) -> None:
        rows = []
        for c in self.criteria:
            all_llm = []
            all_human = []
            for df in self.dataframes.values():
                all_llm.extend(df[c].astype(int).tolist())
                all_human.extend(df[f"{c}_human"].astype(int).tolist())

            all_llm = np.array(all_llm)
            all_human = np.array(all_human)

            cm = confusion_matrix(all_human, all_llm, labels=[0, 1])
            tn, fp, fn, tp = cm.ravel()
            n = len(all_llm)

            rows.append({
                "criterion": c,
                "TP": int(tp),
                "FP": int(fp),
                "FN": int(fn),
                "TN": int(tn),
                "n": n,
                "exact_agreement": round((tp + tn) / n, 3),
                "precision_1": round(tp / (tp + fp), 3) if (tp + fp) > 0 else float("nan"),
                "recall_1":    round(tp / (tp + fn), 3) if (tp + fn) > 0 else float("nan"),
                "precision_0": round(tn / (tn + fn), 3) if (tn + fn) > 0 else float("nan"),
                "recall_0":    round(tn / (tn + fp), 3) if (tn + fp) > 0 else float("nan"),
            })
        self.confusion_matrices = pd.DataFrame(rows).set_index("criterion")

    # ------------------------------------------------------------------
    # 3. Ранговый анализ: method='min' для корректной обработки тай.
    # ------------------------------------------------------------------
    def analyze_per_model(self) -> None:
        llm_ranks   = self.summary["total_llm"].rank(ascending=False, method="min").astype(int)
        human_ranks = self.summary["total_human"].rank(ascending=False, method="min").astype(int)
        self.per_model = pd.DataFrame({
            "total_llm":   self.summary["total_llm"],
            "rank_llm":    llm_ranks,
            "total_human": self.summary["total_human"],
            "rank_human":  human_ranks,
            "rank_delta":  (llm_ranks - human_ranks).abs(),
        })

    def interpret(self) -> str:
        mean_delta = self.per_model["rank_delta"].mean().round(2)
        exact_rank = (self.per_model["rank_delta"] == 0).sum()
        n          = len(self.per_model)
        max_delta  = self.per_model["rank_delta"].max()
        n_ties     = n - self.per_model["rank_human"].nunique()

        if mean_delta == 0:
            level = "полное совпадение ранжирования"
        elif mean_delta <= 0.5:
            level = "высокое совпадение ранжирования"
        elif mean_delta <= 1.0:
            level = "умеренное совпадение ранжирования"
        else:
            level = "слабое совпадение ранжирования"

        result = (
            f"Согласованность LLM-судьи с человеком: {level}. "
            f"Точных совпадений рангов: {exact_rank}/{n}. "
            f"Среднее отклонение: {mean_delta}, максимальное: {max_delta}."
        )
        if n_ties > 0:
            result += (
                f" Примечание: обнаружены тай в human-ранжировании "
                f"({n_ties} моделей делят ранг), method='min'."
            )
        return result

In [ ]:
# =====================================================================
#  Запуск пайплайна
# =====================================================================
output_dir = Path(OUTPUT_DIR)

judge_client = JudgeClient(
    api_url=API_URL,
    api_key=API_KEY,
    model=JUDGE_MODEL,
    max_retries=MAX_RETRIES,
    timeout=TIMEOUT,
    retry_delay=RETRY_DELAY,
)

rescore = RescoringPipeline(
    output_dir=output_dir,
    base_model_files=BASE_MODEL_FILES,
    judge_client=judge_client,
    prompt_template=JUDGE_CRITERIA,
    criteria=CRITERIA,
    version=VERSION,
    sleep_between_requests=0.3,
)
rescore.run()

# =====================================================================
#  Анализ результатов
# =====================================================================
output_dir = Path(OUTPUT_DIR)

loader = DataLoader(
    output_dir=output_dir,
    base_model_files=BASE_MODEL_FILES,
    version=VERSION,
    criteria=CRITERIA,
)
loader.load()
loader.validate()

aggregator = ScoreAggregator(dataframes=loader.dataframes, criteria=CRITERIA, weights=WEIGHTS)
aggregator.build_summary()
aggregator.sort_summary()

analyzer = AgreementAnalyzer(
    dataframes=loader.dataframes,
    summary=aggregator.summary,
    criteria=CRITERIA,
    weights=WEIGHTS,
)
analyzer.analyze_per_criterion()
analyzer.build_confusion_matrices()
analyzer.analyze_per_model()

pd.set_option("display.float_format", "{:.3f}".format)
pd.set_option("display.max_columns", None)

print("=== Средние оценки и итоговая взвешенная оценка ===")
display(aggregator.summary)

print("\n=== Exact agreement (основная), phi и kappa (доп.) по критериям ===")
print("NB: phi/kappa при n≈20 имеют широкие доверительные интервалы, "
      "интерпретировать с осторожностью.\n")
display(analyzer.per_criterion)

print("\n=== Confusion matrices по критериям (агрегация по всем моделям) ===")
print("Positive = 1, Negative = 0. "
      "y_true = human, y_pred = llm.\n")
display(analyzer.confusion_matrices)

print("\n=== Согласованность ранжирования моделей ===")
display(analyzer.per_model)
print(f"\n{analyzer.interpret()}")

# =====================================================================
#  Сохранение
# =====================================================================
output_path = output_dir / f"llm_analysis_results_{VERSION}.xlsx"
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    aggregator.summary.to_excel(writer, sheet_name="summary")
    analyzer.per_criterion.to_excel(writer, sheet_name="per_criterion")
    analyzer.confusion_matrices.to_excel(writer, sheet_name="confusion_matrices")
    analyzer.per_model.to_excel(writer, sheet_name="per_model")
logger.info(f"Saved: {output_path}")

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/')

In [ ]:
import json
import logging
import re
import sys
import time
from getpass import getpass
from pathlib import Path
from typing import Optional

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from sklearn.metrics import cohen_kappa_score
from scipy.stats import pearsonr
from tqdm.auto import tqdm
from urllib3.util.retry import Retry

root_logger = logging.getLogger()
root_logger.handlers.clear()
root_logger.setLevel(logging.INFO)
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
root_logger.addHandler(handler)
logger = logging.getLogger(__name__)

In [ ]:
OUTPUT_DIR = "/content/gdrive/MyDrive/Agile/"
JUDGE_MODEL = "openai/gpt-5.2"
API_URL = "https://api.vsellm.ru/v1/chat/completions"
API_KEY = getpass("API Key: ")

BASE_MODEL_FILES = {
    "gemini-3-pro-preview": "judge_gemini-3-pro-preview",
    "claude-sonnet-4.6":    "judge_anthropic_claude-sonnet-4.6",
    "avibe":                "judge_AvitoTech_avibe",
    "t-lite-it-1.0":        "judge_t-tech_T-lite-it-1.0",
}

CRITERIA = [
    "agile_correctness",
    "practicality",
    "context_handling",
    "politeness",
    "text_cleanliness",
    "language_quality",
]

WEIGHTS = {
    "agile_correctness": 0.1,
    "practicality":      0.3,
    "context_handling":  0.1,
    "politeness":        0.1,
    "text_cleanliness":  0.2,
    "language_quality":  0.2,
}

MAX_RETRIES = 3
TIMEOUT = 120
RETRY_DELAY = 5

In [ ]:
# Единственное место, где меняется версия
VERSION = "v2"

In [ ]:
JUDGE_PROMPTS = {}

JUDGE_PROMPTS["v1"] = """
Ты — строгий проверяющий. Твоя задача — оценить ответ Agile коуча
по 6 критериям. Каждый критерий бинарный: 0 или 1.

### ОСНОВНЫЕ ИНСТРУКЦИИ ###

* Ты действуешь исключительно как **верификатор**, а не как советчик.
  Не исправляй ответ и не предлагай улучшений.
* Проверяй ответ **шаг за шагом** по каждому критерию.
* Для каждого нарушения **цитируй конкретный фрагмент** ответа,
  который стал основанием для оценки 0.
* Не допускай промежуточных значений: только 0 или 1.
  Если есть сомнение — ставь 0.

---

### КРИТЕРИИ ###

**1. agile_correctness — Корректность с точки зрения Agile**

1 — совет полностью соответствует практикам и принципам Agile:
  - каждая упомянутая практика приписана правильной методологии
  - нет утверждений, противоречащих Agile Manifesto или Scrum Guide
  - нет логических ошибок вида "X сломано, значит делай Y"
    без причинно-следственного обоснования

0 — хотя бы одно из:
  - практика приписана неверной методологии
  - совет прямо противоречит базовым принципам Agile
  - причинно-следственная связь между проблемой и рекомендацией
    отсутствует или ложная

---

**2. practicality — Практическая применимость**

1 — ответ содержит конкретное действие, выполнимое в реальной
    корпоративной команде без дополнительных ресурсов:
  - есть понятный следующий шаг
  - рекомендация не требует идеальных условий

0 — хотя бы одно из:
  - совет абстрактный, без конкретных шагов
    (пример нарушения: "улучшите коммуникацию в команде")
  - рекомендация применима только в идеальных условиях
  - непонятно, что именно нужно сделать после прочтения

---

**3. context_handling — Работа с неполным контекстом**

1 — модель корректно обработала уровень полноты вопроса:
  - если контекст достаточен: дан конкретный совет
  - если контекст недостаточен: модель явно обозначила,
    чего не хватает, и либо задала уточняющий вопрос,
    либо дала совет для наиболее вероятного сценария
    с явным указанием допущения

0 — хотя бы одно из:
  - модель сделала неоправданные допущения без их обозначения
  - при явно неполном вопросе дан уверенный совет без оговорок
  - при достаточном контексте модель уклонилась от конкретики

---

**4. politeness — Вежливость и тон**

1 — тон соответствует дружелюбному профессиональному стилю:
  - если в вопросе есть приветствие — ответ начинается
    с ответного приветствия
  - нет покровительственного, холодного или фамильярного тона

0 — хотя бы одно из:
  - сотрудник поздоровался, модель проигнорировала приветствие
  - тон звучит пренебрежительно или менторски
    (пример нарушения: "Вы должны понимать, что...")
  - тон избыточно фамильярный

---

**5. text_cleanliness — Чистота текста**

1 — текст не содержит артефактов:
  - нет эмодзи
  - нет висящих символов (* ** :) в начале текста
  - нет технических артефактов генерации (повторов, обрывов,
    незакрытых скобок)

0 — хотя бы одно из:
  - присутствуют эмодзи
  - текст начинается с ** или :
  - есть артефакты генерации

---

**6. language_quality — Грамотность и языковая чистота**

1 — текст написан грамотно на русском языке:
  - нет орфографических и грубых пунктуационных ошибок
  - англицизмы используются только там, где нет устоявшегося
    русского эквивалента (спринт, бэклог, скрам — допустимо;
    "имплементировать" вместо "внедрить" — нарушение)

0 — хотя бы одно из:
  - есть орфографические или грубые грамматические ошибки
  - текст перегружен английскими словами без необходимости
  - используются слова там, где они неуместны
    (пример нарушения: "эджайл-подход" вместо "Agile-подход")

---

### ИНСТРУКЦИЯ ПО ВЕРИФИКАЦИИ ###

Перед выводом результата выполни следующие шаги:

Шаг 1. Прочитай вопрос сотрудника и финальный ответ модели полностью.

Шаг 2. Для каждого критерия:
  - процитируй фрагмент ответа, на который опираешься
  - зафикси: выполнен критерий или нарушен
  - если нарушен — укажи конкретную причину

Шаг 3. Проверь согласованность оценок: если agile_correctness = 0,
  то practicality, скорее всего, тоже 0 — убедись, что оценки
  не противоречат друг другу.

Шаг 4. Сформируй итоговый JSON.

---

### ФОРМАТ ВЫВОДА ###

Верни результат строго в формате JSON без пояснений вне блока:
```json
{{
  "agile_correctness": <0 или 1>,
  "practicality": <0 или 1>,
  "context_handling": <0 или 1>,
  "politeness": <0 или 1>,
  "text_cleanliness": <0 или 1>,
  "language_quality": <0 или 1>,
  "total": <сумма баллов от 0 до 6>,
  "violations": [
    {{
      "criterion": "<название критерия>",
      "quote": "<цитата из ответа>",
      "reason": "<причина нарушения>"
    }}
  ]
}}
```

Если нарушений нет — "violations" возвращается пустым списком [].

---

### ДАННЫЕ ДЛЯ ОЦЕНКИ ###

Вопрос сотрудника:
{question}

Финальный ответ модели:
{answer}
"""


JUDGE_PROMPTS["v2"] = """
Ты — строгий проверяющий. Твоя задача — оценить ответ Agile-коуча
по 6 критериям. Каждый критерий бинарный: 0 или 1.

### ОСНОВНЫЕ ИНСТРУКЦИИ ###

* Ты действуешь исключительно как **верификатор**, а не как советчик.
  Не исправляй ответ и не предлагай улучшений.
* Проверяй ответ **шаг за шагом** по каждому критерию.
* Для каждого нарушения **цитируй конкретный фрагмент** ответа,
  который стал основанием для оценки 0.
* Не допускай промежуточных значений: только 0 или 1.
  Если есть сомнение — ставь 0.

---

### КРИТЕРИИ ###

**1. agile_correctness — Корректность с точки зрения Agile**

1 — совет в целом соответствует практикам и принципам Agile:
  - упомянутые практики не противоречат Agile Manifesto
    или Scrum Guide
  - нет грубых фактических ошибок (например, приписывание
    практики неверной методологии)
  - рекомендации логически связаны с описанной проблемой

0 — хотя бы одно из:
  - практика приписана неверной методологии
    (например, канбан-практика названа скрам-практикой)
  - совет прямо противоречит базовым принципам Agile
  - рекомендация логически не следует из описанной проблемы

Важно: незначительные неточности в терминологии при верной сути
НЕ являются основанием для 0.

---

**2. practicality — Практическая применимость**

1 — ответ содержит хотя бы одно конкретное действие,
    выполнимое в реальной команде:
  - понятно, что именно нужно сделать
  - рекомендация реалистична для типичной корпоративной среды

0 — хотя бы одно из:
  - совет полностью абстрактный, без единого конкретного шага
    (пример нарушения: «улучшите коммуникацию в команде»
    без указания способа)
  - рекомендация нереалистична или требует идеальных условий
  - совет предметно неверен (agile_correctness = 0)
    И при этом конкретные шаги основаны на этой ошибке

---

**3. context_handling — Работа с контекстом вопроса**

1 — модель дала ответ, адекватный объёму информации в вопросе:
  - при достаточном контексте: дан конкретный совет
  - при неполном контексте: допустимо ЛЮБОЕ из:
    а) задан уточняющий вопрос
    б) дан совет с явным указанием допущения
    в) дан разумный совет для наиболее типичного сценария
       (допущение может быть неявным, если сценарий очевиден)

0 — хотя бы одно из:
  - модель сделала нетипичное или экзотическое допущение
    без его обозначения
  - при достаточном контексте модель уклонилась от ответа
    и ушла в общие фразы
  - ответ не соотносится с тем, что спрашивал сотрудник

Важно: если вопрос допускает несколько разумных трактовок
и модель ответила в рамках одной из них — это НЕ нарушение.
Не требуй от модели перечисления всех возможных сценариев.

---

**4. politeness — Вежливость и тон**

1 — тон соответствует дружелюбному профессиональному стилю:
  - если в вопросе есть приветствие — ответ содержит
    ответное приветствие
  - нет покровительственного, холодного или фамильярного тона

0 — хотя бы одно из:
  - сотрудник поздоровался, модель проигнорировала приветствие
  - тон звучит пренебрежительно или менторски
    (пример нарушения: «Вы должны понимать, что...»)
  - тон избыточно фамильярный

---

**5. text_cleanliness — Чистота текста**

1 — текст не содержит артефактов:
  - нет эмодзи
  - нет висящих символов (* ** :) в начале текста
  - нет технических артефактов генерации (повторов, обрывов,
    незакрытых скобок)

0 — хотя бы одно из:
  - присутствуют эмодзи
  - текст начинается с ** или :
  - есть артефакты генерации

---

**6. language_quality — Грамотность и языковая чистота**

1 — текст написан грамотно на русском языке:
  - нет орфографических и грубых грамматических ошибок
  - профессиональный Agile-жаргон ДОПУСТИМ:
    спринт, бэклог, скрам, стендап, дейли, ретро,
    ревью, фидбэк, митинг, тимлид, продакт-оунер,
    скрам-мастер, стейкхолдер, таск, борд, эстимейт
  - англицизмы недопустимы только когда есть очевидный
    и общеупотребительный русский эквивалент
    (пример нарушения: «имплементировать» вместо «внедрить»,
    «эджайл» вместо «Agile»)

0 — хотя бы одно из:
  - есть орфографические или грубые грамматические ошибки
  - текст перегружен англицизмами сверх профессионального
    жаргона (пример нарушения: «Давайте зашедулим митинг
    для дискаша нашего перформанса»)

Важно: одиночный допустимый англицизм НЕ является основанием
для 0. Оценивай общую читаемость текста, а не отдельные слова.

---

### ИНСТРУКЦИЯ ПО ВЕРИФИКАЦИИ ###

Перед выводом результата выполни следующие шаги:

Шаг 1. Прочитай вопрос сотрудника и финальный ответ модели полностью.

Шаг 2. Для каждого критерия:
  - процитируй фрагмент ответа, на который опираешься
  - зафикси: выполнен критерий или нарушен
  - если нарушен — укажи конкретную причину

Шаг 3. Оцени каждый критерий НЕЗАВИСИМО от остальных.
  Каждая оценка должна быть обоснована собственной цитатой.

Шаг 4. Сформируй итоговый JSON.

---

### ФОРМАТ ВЫВОДА ###

Верни результат строго в формате JSON без пояснений вне блока:
```json
{{
  "agile_correctness": <0 или 1>,
  "practicality": <0 или 1>,
  "context_handling": <0 или 1>,
  "politeness": <0 или 1>,
  "text_cleanliness": <0 или 1>,
  "language_quality": <0 или 1>,
  "total": <сумма баллов от 0 до 6>,
  "violations": [
    {{
      "criterion": "<название критерия>",
      "quote": "<цитата из ответа>",
      "reason": "<причина нарушения>"
    }}
  ]
}}
```

Если нарушений нет — "violations" возвращается пустым списком [].

---

### ДАННЫЕ ДЛЯ ОЦЕНКИ ###

Вопрос сотрудника:
{question}

Финальный ответ модели:
{answer}
"""

JUDGE_PROMPTS["v5"] = """
Ты — строгий проверяющий. Твоя задача — оценить ответ Agile-коуча
по 6 критериям. Каждый критерий бинарный: 0 или 1.

### ОСНОВНЫЕ ИНСТРУКЦИИ ###

* Ты действуешь исключительно как **верификатор**, а не как советчик.
  Не исправляй ответ и не предлагай улучшений.
* Проверяй ответ **шаг за шагом** по каждому критерию.
* Для каждого нарушения **цитируй конкретный фрагмент** ответа,
  который стал основанием для оценки 0.
* Не допускай промежуточных значений: только 0 или 1.
  Если есть сомнение — ставь 0.

---

### КРИТЕРИИ ###

**1. agile_correctness — Корректность с точки зрения Agile**

1 — совет соответствует практикам и принципам Agile:
  - упомянутые практики не противоречат Agile Manifesto
    или Scrum Guide
  - нет фактических ошибок: практики приписаны верным
    методологиям, роли описаны корректно
  - рекомендации логически связаны с описанной проблемой

0 — хотя бы одно из:
  - практика приписана неверной методологии
  - совет противоречит принципам Agile
  - описание ролей или церемоний содержит фактическую ошибку
  - рекомендация логически не следует из описанной проблемы

Примеры фактических ошибок (→ оценка 0):
  - «скрам-мастер назначает задачи разработчикам»
    (нарушение: команда самоорганизующаяся)
  - «ретроспектива проводится в середине спринта»
    (нарушение: ретро — по завершении спринта)
  - «в канбане обязательны двухнедельные итерации»
    (нарушение: канбан не требует фиксированных итераций)
  - «Product Owner отвечает за техническую архитектуру»
    (нарушение: PO отвечает за бизнес-ценность бэклога)

Примеры допустимого (→ НЕ является ошибкой):
  - использование слова «стендап» вместо «Daily Scrum»
  - совет провести ретро, даже если команда не по скраму
    (ретро — общая Agile-практика)
  - нестрогий порядок описания церемоний
  - упрощённое, но верное по сути описание роли

---

**2. practicality — Практическая применимость**

1 — ответ содержит хотя бы одно конкретное действие,
    связанное с ситуацией из вопроса:
  - есть понятный следующий шаг
  - рекомендация реалистична для типичной корпоративной среды
  - действие не обязано быть уникальным — достаточно,
    чтобы оно было релевантно описанной проблеме

0 — хотя бы одно из:
  - совет полностью абстрактный, без единого конкретного шага
    (пример нарушения: «улучшите коммуникацию в команде»
    без указания, как именно)
  - ответ состоит из перечисления общих Agile-практик
    без объяснения, как они помогут в конкретной ситуации
    (пример нарушения: на вопрос «команда не укладывается
    в спринт» ответ перечисляет «проводите ретро, дейли,
    планирование» без привязки к проблеме со сроками)
  - рекомендация нереалистична или требует идеальных условий

Калибровка: если ответ предлагает конкретное действие
и хотя бы кратко объясняет, зачем оно нужно в данной
ситуации — это 1. Просто перечисление практик без связи
с проблемой — это 0.

---

**3. context_handling — Работа с контекстом вопроса**

1 — модель дала ответ, адекватный объёму информации в вопросе:
  - при достаточном контексте: дан конкретный совет
  - при неполном контексте: допустимо ЛЮБОЕ из:
    а) задан уточняющий вопрос
    б) дан совет с явным указанием допущения
    в) дан разумный совет для наиболее типичного сценария
       (допущение может быть неявным, если сценарий очевиден)

0 — хотя бы одно из:
  - модель сделала нетипичное или экзотическое допущение
    без его обозначения
  - при достаточном контексте модель уклонилась от ответа
    и ушла в общие фразы
  - ответ не соотносится с тем, что спрашивал сотрудник

Важно: если вопрос допускает несколько разумных трактовок
и модель ответила в рамках одной из них — это НЕ нарушение.
Не требуй от модели перечисления всех возможных сценариев.

---

**4. politeness — Вежливость и тон**

1 — тон соответствует дружелюбному профессиональному стилю:
  - если в вопросе есть приветствие — ответ содержит
    ответное приветствие
  - нет покровительственного, холодного или фамильярного тона

0 — хотя бы одно из:
  - сотрудник поздоровался, модель проигнорировала приветствие
  - тон звучит пренебрежительно или менторски
    (пример нарушения: «Вы должны понимать, что...»)
  - тон избыточно фамильярный

---

**5. text_cleanliness — Чистота текста**

1 — текст не содержит артефактов:
  - нет эмодзи
  - нет висящих символов (* ** :) в начале текста
  - нет технических артефактов генерации (повторов, обрывов,
    незакрытых скобок)

0 — хотя бы одно из:
  - присутствуют эмодзи
  - текст начинается с ** или :
  - есть артефакты генерации

---

**6. language_quality — Грамотность и языковая чистота**

1 — текст написан грамотно на русском языке:
  - нет орфографических и грубых грамматических ошибок
  - нет нарушений согласования, управления или порядка слов
  - профессиональный Agile-жаргон из следующего списка
    ДОПУСТИМ: спринт, бэклог, скрам, стендап, дейли, ретро,
    ревью, фидбэк, митинг, тимлид, продакт-оунер,
    скрам-мастер, стейкхолдер, таск, борд, эстимейт

0 — хотя бы одно из:
  - есть орфографические или грубые грамматические ошибки
    (согласование, управление, пропущенные слова)
  - используются англицизмы ЗА ПРЕДЕЛАМИ допустимого списка,
    если для них есть общеупотребительный русский эквивалент
    (примеры нарушений: «имплементировать» вместо «внедрить»,
    «коммитмент» вместо «обязательство», «фасилитировать»
    вместо «проводить/модерировать», «апдейт» вместо
    «обновление», «дедлайн» вместо «срок»,
    «эджайл» вместо «Agile»)
  - текст перегружен англицизмами
    (пример нарушения: «Давайте зашедулим митинг
    для дискаша нашего перформанса»)

Калибровка: слова из допустимого списка — норма. Одиночное
заимствование вне списка, но без очевидного русского аналога
(например, «инсайт») — допустимо. Заимствование вне списка,
имеющее устоявшийся русский аналог — нарушение.

---

### ИНСТРУКЦИЯ ПО ВЕРИФИКАЦИИ ###

Перед выводом результата выполни следующие шаги:

Шаг 1. Прочитай вопрос сотрудника и финальный ответ модели полностью.

Шаг 2. Для каждого критерия:
  - процитируй фрагмент ответа, на который опираешься
  - зафикси: выполнен критерий или нарушен
  - если нарушен — укажи конкретную причину

Шаг 3. Оцени каждый критерий НЕЗАВИСИМО от остальных.
  Каждая оценка должна быть обоснована собственной цитатой.

Шаг 4. Сформируй итоговый JSON.

---

### ФОРМАТ ВЫВОДА ###

Верни результат строго в формате JSON без пояснений вне блока:
```json
{{
  "agile_correctness": <0 или 1>,
  "practicality": <0 или 1>,
  "context_handling": <0 или 1>,
  "politeness": <0 или 1>,
  "text_cleanliness": <0 или 1>,
  "language_quality": <0 или 1>,
  "total": <сумма баллов от 0 до 6>,
  "violations": [
    {{
      "criterion": "<название критерия>",
      "quote": "<цитата из ответа>",
      "reason": "<причина нарушения>"
    }}
  ]
}}
```

Если нарушений нет — "violations" возвращается пустым списком [].

---

### ДАННЫЕ ДЛЯ ОЦЕНКИ ###

Вопрос сотрудника:
{question}

Финальный ответ модели:
{answer}
"""

assert VERSION in JUDGE_PROMPTS, f"VERSION='{VERSION}' не найдена. Доступны: {list(JUDGE_PROMPTS)}"
JUDGE_CRITERIA = JUDGE_PROMPTS[VERSION]
logger.info(f"Using prompt version: {VERSION}")

In [ ]:
class JudgeClient:
    def __init__(
        self,
        api_url: str,
        api_key: str,
        model: str,
        max_retries: int = 3,
        timeout: int = 120,
        retry_delay: int = 5,
    ) -> None:
        self.api_url = api_url
        self.api_key = api_key
        self.model = model
        self.max_retries = max_retries
        self.timeout = timeout
        self.retry_delay = retry_delay

    def _create_session(self) -> requests.Session:
        session = requests.Session()
        retry = Retry(
            total=self.max_retries,
            backoff_factor=1,
            status_forcelist=[429, 500, 502, 503, 504],
            allowed_methods=["POST"],
        )
        adapter = HTTPAdapter(max_retries=retry)
        session.mount("http://", adapter)
        session.mount("https://", adapter)
        return session

    def evaluate(self, question: str, answer: str, prompt_template: str) -> Optional[dict]:
        prompt = prompt_template.format(question=question, answer=answer)
        headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {self.api_key}",
        }
        payload = {
            "model": self.model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.0,
        }
        session = self._create_session()
        for attempt in range(self.max_retries):
            try:
                response = session.post(
                    self.api_url, headers=headers, json=payload, timeout=self.timeout,
                )
                response.raise_for_status()
                content = (
                    response.json()
                    .get("choices", [{}])[0]
                    .get("message", {})
                    .get("content", "")
                    .strip()
                )
                return self._parse_json(content)
            except Exception as e:
                logger.warning(f"Attempt {attempt + 1} failed: {e}")
                time.sleep(self.retry_delay * (attempt + 1))
        logger.error("All attempts failed")
        return None

    @staticmethod
    def _parse_json(content: str) -> Optional[dict]:
        clean = re.sub(r"```json|```", "", content).strip()
        try:
            return json.loads(clean)
        except json.JSONDecodeError as e:
            logger.error(f"JSON parse error: {e}. Content: {content[:200]}")
            return None

In [ ]:
class RescoringPipeline:
    def __init__(
        self,
        output_dir: Path,
        base_model_files: dict,
        judge_client: JudgeClient,
        prompt_template: str,
        criteria: list,
        version: str,
    ) -> None:
        self.output_dir = output_dir
        self.base_model_files = base_model_files
        self.judge_client = judge_client
        self.prompt_template = prompt_template
        self.criteria = criteria
        self.version = version

    def _input_path(self, base_name: str) -> Path:
        prev_version = f"v{int(self.version[1:]) - 1}"
        suffix = f"_{prev_version}.xlsx" if int(self.version[1:]) > 1 else ".xlsx"
        return self.output_dir / f"{base_name}{suffix}"

    def _output_path(self, base_name: str) -> Path:
        return self.output_dir / f"{base_name}_{self.version}.xlsx"

    def _empty_scores(self) -> dict:
        return {c: None for c in self.criteria}

    def _evaluate_row(self, question: str, answer: str) -> dict:
        if not isinstance(answer, str) or answer.startswith("[ERROR"):
            return self._empty_scores()
        result = self.judge_client.evaluate(
            question=question,
            answer=answer,
            prompt_template=self.prompt_template,
        )
        if result is None:
            return self._empty_scores()
        return {c: result.get(c) for c in self.criteria}

    def _evaluate_row_majority(self, question: str, answer: str, n_runs: int = 3) -> dict:
        votes = {c: [] for c in self.criteria}
        for _ in range(n_runs):
            scores = self._evaluate_row(question, answer)
            for c in self.criteria:
                if scores[c] is not None:
                    votes[c].append(scores[c])
        return {
            c: int(sum(v) > len(v) / 2) if v else None
            for c, v in votes.items()
        }

    def run(self) -> None:
        for model_name, base_name in self.base_model_files.items():
            input_path = self._input_path(base_name)
            if not input_path.exists():
                logger.warning(f"File not found, skipping {model_name}: {input_path}")
                continue

            df = pd.read_excel(input_path)
            logger.info(f"Loaded {model_name}: {len(df)} rows from {input_path.name}")

            human_cols = [f"{c}_human" for c in self.criteria]
            cols_to_keep = ["question", "reference_answer", "model_answer"] + human_cols
            df_out = df[cols_to_keep].copy()

            new_scores = []
            for _, row in tqdm(df_out.iterrows(), total=len(df_out), desc=model_name):
                # ======================================
                # БЫЛО (одиночный прогон):
                # scores = self._evaluate_row(
                #     question=row["question"],
                #     answer=row["model_answer"],
                # )
                #
                # СТАЛО (majority voting из 3 прогонов):
                scores = self._evaluate_row_majority(
                    question=row["question"],
                    answer=row["model_answer"],
                    n_runs=3,
                )
                # ======================================
                new_scores.append(scores)
                time.sleep(0.5)

            scores_df = pd.DataFrame(new_scores)
            for c in self.criteria:
                df_out.insert(df_out.columns.get_loc(f"{c}_human"), c, scores_df[c])

            output_path = self._output_path(base_name)
            df_out.to_excel(output_path, index=False)
            logger.info(f"Saved: {output_path.name}")

In [ ]:
class DataLoader:
    def __init__(self, output_dir: Path, base_model_files: dict, version: str, criteria: list) -> None:
        self.output_dir = output_dir
        self.base_model_files = base_model_files
        self.version = version
        self.criteria = criteria
        self.dataframes: dict[str, pd.DataFrame] = {}

    def load(self) -> None:
        for model_name, base_name in self.base_model_files.items():
            path = self.output_dir / f"{base_name}_{self.version}.xlsx"
            df = pd.read_excel(path)
            logger.info(f"Loaded {model_name}: {len(df)} rows")
            self.dataframes[model_name] = df

    def validate(self) -> None:
        all_score_cols = self.criteria + [f"{c}_human" for c in self.criteria]
        for model_name, df in self.dataframes.items():
            missing = df[all_score_cols].isnull().sum()
            missing = missing[missing > 0]
            if not missing.empty:
                logger.warning(f"{model_name} missing values:\n{missing.to_string()}")
            else:
                logger.info(f"{model_name}: no missing values")


class ScoreAggregator:
    def __init__(self, dataframes: dict, criteria: list, weights: dict) -> None:
        self.dataframes = dataframes
        self.criteria = criteria
        self.weights = weights
        self.summary: Optional[pd.DataFrame] = None

    def build_summary(self) -> None:
        rows = []
        for model_name, df in self.dataframes.items():
            row = {"model": model_name}
            for c in self.criteria:
                row[f"{c}_llm"]   = df[c].mean().round(3)
                row[f"{c}_human"] = df[f"{c}_human"].mean().round(3)
            row["total_llm"]   = round(sum(row[f"{c}_llm"]   * self.weights[c] for c in self.criteria), 3)
            row["total_human"] = round(sum(row[f"{c}_human"] * self.weights[c] for c in self.criteria), 3)
            rows.append(row)
        self.summary = pd.DataFrame(rows).set_index("model")

    def sort_summary(self) -> None:
        self.summary = self.summary.sort_values("total_human", ascending=False)


class AgreementAnalyzer:
    def __init__(self, dataframes: dict, summary: pd.DataFrame, criteria: list, weights: dict) -> None:
        self.dataframes = dataframes
        self.summary = summary
        self.criteria = criteria
        self.weights = weights
        self.per_criterion: Optional[pd.DataFrame] = None
        self.per_model: Optional[pd.DataFrame] = None

    def analyze_per_criterion(self) -> None:
        rows = []
        for model_name, df in self.dataframes.items():
            row = {"model": model_name}
            for c in self.criteria:
                llm_col   = df[c].astype(int)
                human_col = df[f"{c}_human"].astype(int)
                row[f"{c}_exact"] = (llm_col == human_col).mean().round(3)
                try:
                    phi, _ = pearsonr(llm_col, human_col)
                    row[f"{c}_phi"] = round(phi, 3)
                except Exception:
                    row[f"{c}_phi"] = float("nan")
                try:
                    row[f"{c}_kappa"] = round(cohen_kappa_score(llm_col, human_col), 3)
                except Exception:
                    row[f"{c}_kappa"] = float("nan")
            rows.append(row)
        self.per_criterion = pd.DataFrame(rows).set_index("model")

    def analyze_per_model(self) -> None:
        llm_ranks   = self.summary["total_llm"].rank(ascending=False).astype(int)
        human_ranks = self.summary["total_human"].rank(ascending=False).astype(int)
        self.per_model = pd.DataFrame({
            "total_llm":   self.summary["total_llm"],
            "rank_llm":    llm_ranks,
            "total_human": self.summary["total_human"],
            "rank_human":  human_ranks,
            "rank_delta":  (llm_ranks - human_ranks).abs(),
        })

    def interpret(self) -> str:
        mean_delta = self.per_model["rank_delta"].mean().round(2)
        exact_rank = (self.per_model["rank_delta"] == 0).sum()
        n          = len(self.per_model)
        max_delta  = self.per_model["rank_delta"].max()
        if mean_delta == 0:
            level = "полное совпадение ранжирования"
        elif mean_delta <= 0.5:
            level = "высокое совпадение ранжирования"
        elif mean_delta <= 1.0:
            level = "умеренное совпадение ранжирования"
        else:
            level = "слабое совпадение ранжирования"
        return (
            f"Согласованность LLM-судьи с человеком: {level}. "
            f"Точных совпадений рангов: {exact_rank}/{n}. "
            f"Среднее отклонение: {mean_delta}, максимальное: {max_delta}."
        )

In [ ]:
output_dir = Path(OUTPUT_DIR)

judge_client = JudgeClient(
    api_url=API_URL,
    api_key=API_KEY,
    model=JUDGE_MODEL,
    max_retries=MAX_RETRIES,
    timeout=TIMEOUT,
    retry_delay=RETRY_DELAY,
)

rescore = RescoringPipeline(
    output_dir=output_dir,
    base_model_files=BASE_MODEL_FILES,
    judge_client=judge_client,
    prompt_template=JUDGE_CRITERIA,
    criteria=CRITERIA,
    version=VERSION,
)
rescore.run()

In [ ]:
output_dir = Path(OUTPUT_DIR)

loader = DataLoader(
    output_dir=output_dir,
    base_model_files=BASE_MODEL_FILES,
    version=VERSION,
    criteria=CRITERIA,
)
loader.load()
loader.validate()

aggregator = ScoreAggregator(dataframes=loader.dataframes, criteria=CRITERIA, weights=WEIGHTS)
aggregator.build_summary()
aggregator.sort_summary()

analyzer = AgreementAnalyzer(
    dataframes=loader.dataframes,
    summary=aggregator.summary,
    criteria=CRITERIA,
    weights=WEIGHTS,
)
analyzer.analyze_per_criterion()
analyzer.analyze_per_model()

pd.set_option("display.float_format", "{:.3f}".format)
pd.set_option("display.max_columns", None)

print("=== Средние оценки и итоговая взвешенная оценка ===")
display(aggregator.summary)
print("\n=== Точное совпадение, phi и kappa по критериям ===")
display(analyzer.per_criterion)
print("\n=== Согласованность ранжирования моделей ===")
display(analyzer.per_model)
print(f"\n{analyzer.interpret()}")

output_path = output_dir / f"llm_analysis_results_{VERSION}.xlsx"
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    aggregator.summary.to_excel(writer, sheet_name="summary")
    analyzer.per_criterion.to_excel(writer, sheet_name="per_criterion")
    analyzer.per_model.to_excel(writer, sheet_name="per_model")
logger.info(f"Saved: {output_path}")